# [실습] 회귀 분석

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 📈 무엇이 결과를 움직이나 — 회귀분석

## — 상관에서 회귀로, 그리고 "통제된 효과"까지

---

이전 시간에는 "차이가 있나"를 따졌습니다. 오늘은 한 걸음 더 나아가 **"무엇이 무엇과 함께 움직이고, 무엇이 무엇을 설명하는가"** — 변수들 사이의 *관계*를 다룹니다. 데이터 분석에서 가장 많이 쓰이는 도구, **회귀분석(Regression)**의 세 가지 얼굴을 만나봅니다.

## 📋 학습 목표

이 노트북을 끝내면 여러분은 다음을 할 수 있습니다.

1. **상관계수**(Pearson·Spearman)로 두 변수의 관계 방향·세기를 재고, 그 한계를 설명할 수 있습니다.
2. **"상관은 인과가 아니다"**의 의미를 교란변수(Confounder) 사례로 설명할 수 있습니다.
3. `statsmodels`로 **단순·다중 회귀**를 적합하고 계수·신뢰구간·F-검정·R²를 해석할 수 있습니다.
4. **다중공선성(VIF)·잔차 진단·이상치 영향**을 점검해 모델의 신뢰성을 평가할 수 있습니다.
5. 회귀계수를 단순한 숫자가 아니라 **"다른 변수를 통제한 조건에서의 평균 효과"**로 신중히 해석할 수 있습니다.

## 📚 학습 목차

| Part | 주제 | 이 파트에서 답할 질문 | ⏱ 예상 |
|------|------|----------------------| --- |
| 1 | 왜 회귀인가 | 차이를 넘어, 관계는 어떻게 다루나? | 5분 |
| 2 | 환경 준비와 오늘의 데이터 | 무엇을, 어떤 도구로 다루나? | 10분 |
| 3 | 상관 — 함께 움직이는가 | 두 변수의 관계를 어떻게 한 숫자로 재나? | 40분 |
| 4 | 상관 ≠ 인과 | 함께 움직인다고 원인일까? | 10분 |
| 5 | 단순 회귀 | 한 변수로 결과를 어떻게 설명하나? | 30분 |
| 6 | 다중 회귀 | 여러 변수를 한꺼번에 넣으면? | 30분 |
| 7 | 회귀 진단 | 이 회귀, 믿어도 되나? | 40분 |
| 8 | 회귀의 세 얼굴 | 회귀는 결국 무엇을 위한 도구인가? | 15분 |
| 🧪 | 종합 실습 | 실제 데이터로 회귀를 적합·진단·보고할 수 있나? | 40분 |

*⏱ 예상 시간은 참고용 안내입니다. 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다.*

# Part 1. 왜 회귀인가 — 차이를 넘어 관계로

지금까지의 여정을 잠깐 돌아볼까요. 앞서 데이터 한 덩어리의 *생김새*(분포)를 봤고, 지금까지는 두 집단·여러 집단의 *차이*를 검정했습니다. "남녀 결제액이 다른가?", "종에 따라 체중이 다른가?" 같은 질문이었습니다.

그런데 현실의 질문은 대개 더 복잡합니다. "광고비를 **늘리면** 매출이 **얼마나** 오를까?", "공부 시간이 시험 점수와 **어떤 관계**일까?", "나이·BMI·혈압 중 **무엇이** 질병 진행을 가장 잘 설명할까?" 이건 '차이'가 아니라 **'관계'**를 묻는 질문입니다. 한 변수가 변할 때 다른 변수가 *어떻게, 얼마나* 변하는지를 알고 싶은 겁니다.

이 질문에 답하는 핵심 도구가 **회귀분석**입니다. 회귀는 변수들 사이의 관계를 하나의 식(보통 직선)으로 요약해, *예측*하고 *설명*하고 때로는 *효과를 추정*하게 해줍니다. 데이터 분석가·데이터 사이언티스트가 가장 자주 손에 쥐는 도구이자, 머신러닝의 출발점이기도 합니다.

> ❓ **이 파트에서 답할 질문:** 변수 사이의 '관계'는 어떻게 숫자와 식으로 표현하고, 회귀는 그 관계로 무엇을 할 수 있을까요?

> 🎯 **오늘의 핵심:** 회귀계수는 그냥 숫자가 아니라 **"다른 조건이 같을 때, 이 변수가 1 늘면 결과가 얼마 변하는가"**라는 *조건부 이야기*입니다. 그리고 — **상관(함께 움직임)은 인과(원인)가 아닙니다.** 이 두 문장이 오늘 내내 따라다닐 겁니다.

오늘은 관계를 보는 가장 단순한 도구인 *상관*에서 출발해, 한 변수로 설명하는 *단순 회귀*, 여러 변수를 함께 보는 *다중 회귀*, 그 회귀가 믿을 만한지 따지는 *진단*, 그리고 회귀가 쓰이는 세 가지 방식까지 차근차근 올라갑니다. 회귀는 한 번에 이해되는 도구가 아니니, 각 Part의 작은 질문 하나씩에만 집중하며 따라오세요. 오늘의 여정을 한눈에 그리면 이렇습니다 — *상관*(함께 움직이나?) → *상관 ≠ 인과*(착각 조심) → *단순 회귀*(한 변수로 설명) → *다중 회귀*(여러 변수를 통제) → *진단*(믿을 만한가?) → *세 얼굴*(예측·검정·인과). 각 단계가 바로 앞 단계의 한계를 메우며 자연스럽게 이어지니, 큰 흐름만 잡고 가도 충분합니다.

# Part 2. 환경 준비와 오늘의 데이터

라이브러리를 불러오고, 오늘 사용할 실제 데이터를 준비합니다. 회귀는 `statsmodels`를 주력으로 씁니다 — 계수·p값·신뢰구간·진단 통계를 풍부하게 보여주기 때문입니다.

> ⚠️ **주의:** 코드 셀은 반드시 위에서부터 순서대로 실행하세요. 중간부터 실행하면 이전 셀의 변수를 찾지 못해 오류가 납니다.

## 1단계 · 환경 준비 — 라이브러리·한글 폰트·시드 고정

In [ ]:
# ─────────────────────────────────────────────
# [C1] 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scipy statsmodels matplotlib seaborn scikit-learn -q

import platform
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf   # 수식(formula)으로 회귀를 쓰는 인터페이스
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정

# 한글 폰트 설정
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    # 리눅스·Colab에서 한글이 □□로 보이면 아래를 먼저 실행하세요.
    # !apt-get -qq install -y fonts-nanum && rm -rf ~/.cache/matplotlib
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료!")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("scipy :", scipy.__version__)

## 📖 핵심 용어 사전

| 기호 | 읽는 법 | 의미 |
|------|---------|------|
| $y$ | 종속변수 (반응변수) | 설명하려는 결과 |
| $x$ | 독립변수 (설명변수·예측변수) | 결과를 설명하는 입력 |
| $\hat{y}$ | 와이 햇 | 회귀식이 예측한 값 |
| $\beta_0,\ \beta_1$ | 베타 | 절편·기울기(회귀계수) |
| $\varepsilon$ | 엡실론 | 오차항(식으로 설명 못한 나머지) |
| $r$ | — | 상관계수 ($-1 \le r \le 1$) |
| $R^2$ | 결정계수 | 회귀가 설명한 변동의 비율(0~1) |
| VIF | — | 분산팽창계수(다중공선성 지표) |

> 회귀의 기본형: $\;y = \beta_0 + \beta_1 x + \varepsilon$. "결과 = 절편 + 기울기×입력 + 오차"라고 읽습니다.

## 오늘 쓸 실제 데이터

네 가지 실제 데이터를 씁니다. **모두 라이브러리가 함수 하나로 제공**합니다.

| 데이터 | 어디서 오나 | 오늘의 쓰임 |
| --- | --- | --- |
| **diabetes** | scikit-learn **내장** | 환자 442명의 10개 지표 → *질병 진행도* 설명. **다중 회귀의 메인 무대** |
| **engel** | statsmodels **내장** | 가계 235곳의 소득 → 식료품 지출. **단위가 있는 단순 회귀**·이분산 실사례 |
| **CASchools** | statsmodels → Rdatasets(`AER`) | 학군 420곳의 시험 점수·학생·소득 → **"통제"의 위력**을 보여주는 다중 회귀 |
| **longley** | statsmodels **내장** | 거시경제 7개 변수 → **VIF(다중공선성) 교과서 예제** |

- `diabetes`의 설명변수는 **미리 표준화**돼 있습니다. 덕분에 계수 크기로 *상대적 중요도*를 비교하기 좋지만, 대신 "1 늘면"의 단위가 추상적입니다.
- 그래서 **단위가 살아 있는 데이터**를 함께 씁니다. `engel`은 소득(벨기에 프랑) → 식료품 지출(프랑)이라 *"소득이 100 늘면 지출이 몇 늘까"* 를 곧바로 읽을 수 있습니다.
- `CASchools`는 학군별 **학생-교사비**와 **시험 점수** 데이터입니다. 단순 회귀와 다중 회귀의 계수가 극적으로 달라지는 사례라, "통제한다"는 말의 의미를 데이터로 체감할 수 있습니다.

> ⚠️ `sm.datasets.get_rdataset()`은 공개 저장소(Rdatasets)에서 자동으로 받아오므로 **첫 실행에 인터넷 연결이 필요**합니다. `load_diabetes`·`engel`·`longley`는 패키지에 함께 설치돼 있어 오프라인에서도 열립니다.

## 2단계 · 실습 데이터 4종 불러오기

In [ ]:
# [C2] ── 데이터 로드 — 모두 라이브러리가 제공하는 데이터셋 ─────────────
from sklearn.datasets import load_diabetes

# (1) scikit-learn 내장: 질병 진행도 (설명변수는 표준화되어 있음)
dia = load_diabetes(as_frame=True)
df = dia.frame                       # 컬럼: age sex bmi bp s1..s6 target

# (2) statsmodels 내장: 소득 → 식료품 지출 (단위가 살아 있는 데이터)
engel = sm.datasets.engel.load_pandas().data          # income, foodexp (벨기에 프랑)

# (3) statsmodels 내장: 거시경제 (다중공선성 교과서)
longley = sm.datasets.longley.load_pandas().data

# (4) Rdatasets: 캘리포니아 학군 420곳
ca = sm.datasets.get_rdataset("CASchools", "AER").data.copy()
ca["testscr"] = (ca["read"] + ca["math"]) / 2          # 읽기·수학 평균 점수 (결과 변수)
ca["str_ratio"] = ca["students"] / ca["teachers"]      # 학생-교사비 (교사 1명당 학생 수)

print("diabetes :", df.shape, "| 컬럼:", list(df.columns))
print("engel    :", engel.shape, "| 컬럼:", list(engel.columns),
      f"| 소득 평균 {engel['income'].mean():.0f}, 식료품 지출 평균 {engel['foodexp'].mean():.0f}")
print("CASchools:", ca.shape, "| 점수 평균 %.1f, 학생-교사비 평균 %.1f"
      % (ca["testscr"].mean(), ca["str_ratio"].mean()))
print("longley  :", longley.shape, "| 컬럼:", list(longley.columns))
print()
print("diabetes 미리보기:")
print(df[["age", "bmi", "bp", "s5", "target"]].head().round(3))

## 🚦 짚고 넘어가기 — Part 2 환경과 데이터

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. 회귀식 $y = \beta_0 + \beta_1 x + \varepsilon$ 에서 $\beta_1$과 $\varepsilon$은 각각 무엇을 뜻하나요?
2. $R^2$는 무엇을 0~1 사이로 나타내나요?
3. '관계'를 묻는 질문과 '차이'를 묻는 질문의 예를 하나씩 들어보세요.

> 💡 **다음 Part 예고:** 회귀로 들어가기 전에, 관계를 보는 가장 단순한 한 숫자 — **상관계수**부터 익힙니다. Part 3으로 갑니다.

# Part 3. 상관 — 두 변수가 함께 움직이는가

회귀의 출발점은 **상관(Correlation)**입니다. "한 변수가 커질 때 다른 변수도 커지는가(또는 작아지는가)"를 *하나의 숫자*로 요약한 것입니다.

> ❓ **이 파트에서 답할 질문:** 두 변수가 얼마나, 어느 방향으로 함께 움직이는지를 하나의 숫자로 어떻게 잴까요? 그 숫자의 함정은 무엇일까요?

## 💡 쉽게 말하면 — 키와 몸무게처럼 함께 움직이는가

키와 몸무게를 떠올려 보세요. 키가 큰 사람이 대체로 몸무게도 더 나갑니다. 이렇게 "한쪽이 크면 다른 쪽도 큰" 경향을 **양의 상관**, "한쪽이 크면 다른 쪽은 작은" 경향(예: 운동량과 체지방률)을 **음의 상관**이라 합니다. 상관계수 $r$은 이 경향의 *방향*(부호)과 *세기*(절댓값)를 $-1$에서 $+1$ 사이 한 숫자로 압축합니다.

$r = +1$이면 완벽한 우상향 직선, $r = -1$이면 완벽한 우하향 직선, $r = 0$이면 *직선적* 관계가 없다는 뜻입니다.

아래 그림은 **여러 산점도와 그에 해당하는 상관계수**를 모아둔 것입니다. 특히 맨 아래 줄을 잘 보세요 — **뚜렷한 패턴이 있는데도 $r = 0$** 인 경우들입니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/d/d4/Correlation_examples2.svg" width="680" alt="Correlation examples2.svg">

여러 산점도의 상관계수. **첫 줄**은 관계의 세기(−1에서 1까지), **둘째 줄**은 기울기가 달라도 직선이면 r=±1인 것(맨 오른쪽처럼 기울기가 0이면 정의되지 않음), **셋째 줄**은 강한 비선형 관계인데도 r이 0으로 나오는 예입니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Correlation examples2.svg` (CC0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Correlation_examples2.svg

> 💡 **셋째 줄이 오늘의 경고입니다.** 상관계수는 오직 **직선** 관계만 잡아냅니다. 그러니 $r \approx 0$을 "관계 없음"으로 읽으면 안 되고, 반드시 산점도를 함께 봐야 합니다.

## 두 가지 상관계수 — Pearson과 Spearman

| 구분 | Pearson $r$ | Spearman $\rho$ |
|------|-------------|-----------------|
| 무엇을 보나 | **선형(직선)** 관계의 세기 | **순위(단조)** 관계의 세기 |
| 입력 | 값 그대로 | 값을 순위로 바꿔서 |
| 강한 곳 | 직선 관계, 정규분포 | 곡선이라도 *단조 증가/감소*면 OK, 이상치에 강함 |
| 언제 | 선형·연속·정규 | 비선형(단조)·서열·이상치 우려 |

> Pearson은 "직선에 얼마나 가까운가", Spearman은 "순위가 얼마나 함께 오르내리는가"를 봅니다. 앞서 배운 모수 vs 비모수 구도와 정확히 닮았습니다 — 값이냐 순위냐.

## 예제 · BMI와 질병 진행도의 Pearson·Spearman 상관

In [ ]:
# [C3] diabetes: bmi와 질병 진행도(target)의 상관 — Pearson vs Spearman
x = df["bmi"]; y = df["target"]
r_p, p_p = stats.pearsonr(x, y)                                   # Pearson 상관 — '직선' 관계의 세기 (-1~1)
r_s, p_s = stats.spearmanr(x, y)                                  # Spearman 상관 — 순위 기반, 곡선이어도 단조면 잡아냄
print(f"Pearson  r = {r_p:.3f}  (p = {p_p:.2e})   선형 관계의 세기")
print(f"Spearman ρ = {r_s:.3f}  (p = {p_s:.2e})   순위 관계의 세기")

fig, ax = plt.subplots(figsize=(7, 4.5))                          # 그래프 하나를 그릴 도화지 준비
sns.regplot(x=x, y=y, ax=ax, scatter_kws=dict(alpha=0.4, s=20), line_kws=dict(color="indianred"))  # 산점도 + 회귀선을 한 번에
ax.set_xlabel("BMI (standardized)"); ax.set_ylabel("disease progression")
ax.set_title(f"BMI vs target  (Pearson r = {r_p:.2f})")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 점들이 우상향으로 흩어져 있고, 빨간 회귀선이 그 경향을 요약합니다. BMI가 높을수록 질병 진행도가 높아지는 *양의 상관*입니다. Pearson과 Spearman이 비슷하다면 관계가 대체로 직선적이라는 뜻입니다.

## 예제 · 상관계수의 함정 — 곡선과 이상치

In [ ]:
# [C4] 상관계수의 함정 — 같은 r=0 근처라도 관계는 천차만별 (개념 시연)
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정
n = 200
x1 = np.random.uniform(-3, 3, n); y1 = 2*x1 + np.random.normal(0, 1.5, n)   # 선형
x2 = np.random.uniform(-3, 3, n); y2 = x2**2 + np.random.normal(0, 1, n)    # 곡선(U자)
x3 = np.random.uniform(-3, 3, n); y3 = np.random.normal(0, 1, n)            # 무관계
x3[0], y3[0] = 12, 12                                                       # 이상치 1개

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))                 # 그림 한 장에 그래프 3개(1행 3열)
for ax, xx, yy, t in zip(axes, [x1, x2, x3], [y1, y2, y3],
                         ["linear", "curved (U)", "no relation + 1 outlier"]):
    ax.scatter(xx, yy, alpha=0.4, s=15)
    ax.set_title(f"{t}\nPearson r = {stats.pearsonr(xx, yy)[0]:.2f}")  # Pearson 상관 — '직선' 관계의 세기 (-1~1)
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 가운데 U자 그래프는 *뚜렷한 관계*가 있는데도 Pearson r은 0에 가깝습니다 — Pearson은 *직선* 관계만 보기 때문입니다. 오른쪽은 *이상치 한 점*이 상관을 인위적으로 만들어냅니다. **상관계수 하나만 믿지 말고 반드시 산점도를 함께 보라**는 교훈입니다.

### 📌 고전 사례 — 앤스컴의 4중주(Anscombe's quartet)

이 교훈을 가장 유명하게 보여준 예가 1973년 통계학자 앤스컴이 만든 네 데이터입니다. 아래 네 산점도는 **평균·분산·상관계수·회귀직선이 모두 (거의) 똑같습니다.** 그런데 모양은 전혀 다릅니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/e/ec/Anscombe%27s_quartet_3.svg" width="660" alt="Anscombe's quartet 3.svg">

앤스컴의 4중주. 네 데이터 모두 x·y의 평균과 분산, 상관계수(r≈0.816), 회귀직선($y = 3 + 0.5x$)이 동일합니다. 그러나 ① 평범한 선형, ② 곡선, ③ 직선 + 이상치 1개, ④ 한 점이 전체를 지배 — 실제 구조는 완전히 다릅니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Anscombe's quartet 3.svg` (CC BY-SA 3.0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Anscombe%27s_quartet_3.svg

> 💡 **그래서 순서는 늘 이렇습니다:** *산점도를 먼저 보고 → 상관계수를 계산하고 → 회귀를 적합한다.* 숫자부터 보면 ②③④를 ①로 착각합니다. `seaborn`에는 이 데이터가 들어 있어 직접 확인할 수 있습니다 — `sns.load_dataset("anscombe")`.

> 📌 **다른 산업에서는?** 상관 분석은 마케팅(광고 채널 간 관계), 금융(자산 간 상관으로 분산투자), 제조(공정 변수 간 관계), 헬스케어(증상-지표 관계) 어디서나 탐색의 첫걸음으로 쓰입니다.

## 그림으로 · 상관 행렬 히트맵

In [ ]:
# [C5] 여러 변수의 상관을 한눈에 — 상관 행렬 히트맵 (이해 후 탐색이므로 seaborn)
corr = df[["age", "bmi", "bp", "s1", "s3", "s5", "target"]].corr()
fig, ax = plt.subplots(figsize=(7.5, 6))                          # 그래프 하나를 그릴 도화지 준비
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)  # 상관 행렬을 색으로 — 진할수록 강한 상관
ax.set_title("Correlation matrix (diabetes)")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 빨강은 양의 상관, 파랑은 음의 상관, 진할수록 강합니다. `target` 행을 보면 bmi·bp·s5가 질병 진행도와 비교적 강한 양의 상관을 보입니다. 이렇게 히트맵으로 *어떤 변수가 결과와 관련 깊은지* 먼저 감을 잡고 회귀로 넘어갑니다. (단, s1·s3처럼 변수끼리도 강하게 상관된 게 보이면 — 나중에 다중공선성 문제로 이어집니다.)

## Spearman이 빛나는 순간 — 곡선이지만 단조로운 관계

Pearson이 곡선 관계에서 약하다고 했습니다. 그런데 *곡선이어도 한 방향으로만 꾸준히 증가(또는 감소)하는 단조* 관계라면, **Spearman**이 그 관계를 훨씬 잘 잡아냅니다. 순위만 보기 때문에 "휘었어도 계속 오르기만 하면" 강한 상관으로 읽거든요.

In [ ]:
# [C6] 단조-비선형 관계: Pearson은 과소평가, Spearman은 제대로 잡는다
np.random.seed(0)                                                 # 재현성: 난수를 항상 같게 고정
x = np.sort(np.random.uniform(0, 5, 150))
y = np.exp(x) + np.random.normal(0, 3, 150)     # 지수적으로 증가 (곡선이지만 단조 증가)
print(f"Pearson  r = {stats.pearsonr(x, y)[0]:.3f}   (곡선이라 다소 과소평가)")  # Pearson 상관 — '직선' 관계의 세기 (-1~1)
print(f"Spearman ρ = {stats.spearmanr(x, y)[0]:.3f}   (단조 증가를 강하게 포착)")  # Spearman 상관 — 순위 기반, 곡선이어도 단조면 잡아냄
fig, ax = plt.subplots(figsize=(6.5, 4)); ax.scatter(x, y, alpha=0.5, s=15)  # 그래프 하나를 그릴 도화지 준비
ax.set_title("Monotonic but curved: Spearman > Pearson"); plt.tight_layout(); plt.show()  # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 관계가 위로 휜 곡선이지만 *계속 증가*합니다. Pearson은 "직선"이 아니라서 상관을 다소 낮게 보고하지만, Spearman은 순위로 보기에 거의 완벽한 단조 관계로 읽습니다. **"곡선이지만 한 방향"이면 Spearman을 떠올리세요.**

## 상관과 회귀는 어떻게 이어지나 — 그리고 상관계수의 함정 둘

상관계수와 회귀 기울기는 사실 **한 뿌리**입니다. 단순 회귀의 기울기 $\beta_1$과 상관계수 $r$ 사이에는 $\beta_1 = r \times (s_y / s_x)$ 라는 깔끔한 관계가 있어요($s_x, s_y$는 각 변수의 표준편차). 즉 둘 다 "함께 움직이는 정도"를 재지만, $r$은 *단위 없는* 표준화된 값($-1$~$1$)이고 $\beta_1$은 *실제 단위*를 입은 값입니다. 그래서 상관은 "관계가 있나, 얼마나 강한가"를 빠르게 보고, 회귀는 "x가 1 늘면 y가 *몇 단위* 변하나"까지 말해줍니다.

한 가지 더 — **상관은 대칭이지만 회귀는 비대칭**입니다. "x와 y의 상관"과 "y와 x의 상관"은 똑같지만, "x로 y를 예측하는 회귀"와 "y로 x를 예측하는 회귀"는 서로 다른 직선을 냅니다. 무엇을 *결과(y)*로 둘지는 분석의 질문이 정하는 겁니다.

상관계수를 쓸 때 꼭 기억할 함정 두 가지를 정리합니다. **첫째, 상관계수에도 p값이 있습니다.** 표본이 작으면 꽤 큰 $r$도 "우연일 수 있음"(유의하지 않음)으로 나오고, 표본이 매우 크면 $r=0.05$ 같은 *실질적으로 미미한* 상관도 "통계적으로 유의"하게 나옵니다 — 앞서 배운 그 교훈이 여기서도 반복됩니다. 그래서 $r$의 *크기*(실질적 의미)와 *유의성*(p)을 함께 봅니다. **둘째, 상관계수는 오직 직선 관계만 봅니다.** 앞 그림의 U자처럼 뚜렷한 곡선 관계도 $r$은 0에 가깝게 보고하므로, 숫자 하나로 "관계 없음"을 단정하면 안 됩니다. 늘 산점도와 함께 보세요.

## ⌨️ 백문이 불여일타! (1) — bp와 target의 상관·산점도

```
[문제]
1) diabetes에서 혈압(bp)과 질병 진행도(target)의 Pearson·Spearman 상관을 구해보세요.
2) 산점도(regplot)로 두 변수의 관계를 눈으로 확인해보세요.
```

In [ ]:
# [C7] ⌨️ 백문이 불여일타! (1) — bp와 target의 상관·산점도 확인

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C3]에서 `bmi`로 했던 흐름 그대로입니다 — `x`를 `df["bp"]`로 바꿔보세요.
- 상관계수는 `stats.pearsonr(x, y)`와 `stats.spearmanr(x, y)`로 구합니다.
- 산점도는 `sns.regplot(x=x, y=y)`를 쓰면 회귀선까지 한 번에 그려줍니다.

</details>

## 🚦 짚고 넘어가기 — Part 3 상관

1. Pearson과 Spearman 상관계수는 각각 무엇(선형/순위)을 재나요?
2. 상관계수가 0에 가까워도 "관계가 없다"고 단정하면 안 되는 이유는?
3. 상관 분석에서 산점도를 꼭 함께 봐야 하는 이유는 무엇인가요?

> 💡 **다음 Part 예고:** 상관이 강하다고 한 변수가 다른 변수의 *원인*일까요? 가장 흔하고 위험한 오해 — **"상관 ≠ 인과"**를 Part 4에서 짚습니다.

# Part 4. 상관 ≠ 인과 — 함께 움직인다고 원인일까

통계에서 가장 자주, 가장 비싸게 저지르는 실수가 이겁니다. **두 변수가 강하게 상관한다고 해서, 한쪽이 다른 쪽의 *원인*인 것은 아닙니다.**

> ❓ **이 파트에서 답할 질문:** 강한 상관을 보고 "A가 B의 원인"이라 말하면 왜 위험할까요? 무엇이 우리를 속이는 걸까요?

## 💡 쉽게 말하면 — 아이스크림과 익사 사고

여름이면 아이스크림 판매량이 늘고, 동시에 물놀이 사고(익사)도 늘어납니다. 둘은 강하게 상관합니다. 그렇다고 "아이스크림이 익사를 유발한다"고 말하면 우스꽝스럽습니다. 진짜 원인은 따로 있습니다 — **기온(여름)**입니다. 더우니까 아이스크림도 더 팔리고, 물에도 더 많이 들어가는 겁니다.

이렇게 *겉으로 드러나지 않은 제3의 변수*가 두 변수를 동시에 움직여 만드는 가짜 관계를 **허위상관(Spurious Correlation)**, 그 제3의 변수를 **교란변수(Confounder)**라고 부릅니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/a/a6/Confounding.svg" width="520" alt="Confounding.svg">

교란의 구조. 교란변수(Z)가 원인 후보(X)와 결과(Y)에 **모두** 화살표를 보냅니다. 이 구조에서는 X와 Y 사이에 직접 화살표가 없어도 둘이 함께 움직입니다 — 그것이 허위상관입니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Confounding.svg` (CC BY 4.0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Confounding.svg

> 💡 **화살표로 생각하는 습관:** "X → Y"인가, "X ← Z → Y"인가, 아니면 "X ← Y"(역인과)인가. 이 세 그림 중 어느 것인지 묻는 것만으로도 성급한 인과 주장을 크게 줄일 수 있습니다. 이 그림 언어를 본격적으로 쓰는 것이 **인과 그래프(DAG)** 이고, D016·D017의 배경 언어입니다.

## 상관이 인과가 아닌 세 가지 경우

1. **교란변수 (제3의 공통 원인):** 기온 → (아이스크림, 익사). 위 예시.
2. **역인과 (방향이 거꾸로):** "병원에 자주 가는 사람이 더 아프다" — 아파서 병원에 가는 것이지, 병원이 아프게 한 게 아닙니다.
3. **우연 (표본의 장난):** 변수를 충분히 많이 비교하면, 아무 의미 없이도 우연히 강하게 상관하는 쌍이 나옵니다.

> 💡 **그래서 무엇이 필요한가?** 인과를 말하려면 *다른 조건을 같게 만들고(통제)* 비교해야 합니다. 그 통제의 한 도구가 **다중 회귀**(교란변수를 함께 넣어 통제)이고, 가장 강력한 방법이 **무작위 실험(A/B 테스트)** 입니다 — 이후 배울 주제입니다.

이게 왜 *비싼* 실수인지 실무 예로 들어보겠습니다. 한 회사가 "이메일을 연 고객이 그렇지 않은 고객보다 훨씬 많이 구매한다"는 강한 상관을 발견하고, 모든 고객에게 이메일을 퍼붓는 캠페인에 큰 예산을 투입합니다. 그런데 진실은 따로 있었습니다 — *원래 브랜드에 관심이 많은 고객*이 이메일도 더 열고 구매도 더 했던 겁니다. 여기서 '관심도'가 교란변수입니다. 이메일 자체의 효과는 생각보다 미미했고, 예산은 상당 부분 낭비됐습니다. '상관을 인과로 착각한' 전형적인 대가입니다.

비슷한 함정은 곳곳에 있습니다. "운동을 많이 하는 사람이 더 건강하다"는 상관에는 *역인과*(건강해서 운동을 더 할 수 있는 것)가 숨어 있을 수 있고, "아이의 발이 클수록 시험 성적이 좋다"는 *나이*라는 공통 원인이 만든 허위상관입니다. 심지어 변수를 충분히 많이 비교하면, 의미가 전혀 없는데도 우연히 강하게 상관하는 쌍이 반드시 나타납니다.

그래서 데이터로 *의사결정*을 내릴 때는, 결과를 보고 곧장 행동하기 전에 늘 멈춰 물어야 합니다 — **"이 관계가 정말 인과일까, 아니면 숨은 제3의 변수가 만든 착시일까? 방향이 거꾸로일 가능성은? 그저 우연일 가능성은?"** 이 한 번의 질문이 큰 손실을 막고, 여러분을 '숫자를 읽는 사람'에서 '숫자를 의심할 줄 아는 사람'으로 끌어올립니다. 그리고 그 의심에 체계적으로 답하는 방법이 바로 다음 시간의 인과추론입니다.

## 시뮬레이션 · 교란변수 Z가 만든 허위상관

In [ ]:
# [C8] 허위상관 시뮬레이션 — 교란변수 Z가 X와 Y를 동시에 움직인다 (개념 시연)
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정
n = 300
Z = np.random.normal(0, 1, n)            # 숨은 공통 원인 (예: 기온)
X = 2*Z + np.random.normal(0, 0.5, n)    # 아이스크림 판매  (Z가 원인)
Y = 3*Z + np.random.normal(0, 0.5, n)    # 익사 사고 수      (Z가 원인)
# X와 Y 사이에는 '직접' 인과가 없지만...
r_xy = stats.pearsonr(X, Y)[0]                                    # Pearson 상관 — '직선' 관계의 세기 (-1~1)
print(f"X(아이스크림)와 Y(익사)의 상관 r = {r_xy:.3f}  ← 매우 강함!")
print("그러나 X→Y 직접 인과는 없습니다. 둘 다 Z(기온)가 움직인 결과일 뿐.")

# Z의 구간별로 나눠 보면 X-Y 관계가 사라진다 (Z를 '통제'하면)
g = pd.qcut(Z, 3, labels=["저", "중", "고"])                         # 값을 같은 크기의 구간으로 나눠 범주화
for lvl in ["저", "중", "고"]:
    mask = (g == lvl)
    print(f"  Z={lvl} 구간 안에서 X-Y 상관 = {stats.pearsonr(X[mask], Y[mask])[0]:.3f}")  # Pearson 상관 — '직선' 관계의 세기 (-1~1)

> **읽는 법:** 전체로 보면 X와 Y는 강하게 상관(r이 큽니다)하지만, **Z를 비슷한 값끼리 묶어(통제) 그 안에서만 보면 상관이 거의 사라집니다.** 이게 "Z가 만든 허위상관"의 정체입니다. 회귀에 Z를 *함께 넣어 통제*하면 같은 일을 할 수 있습니다 — Part 6에서 봅니다.

> 📌 **다른 산업에서는?** 마케팅(접속이 많은 고객이 매출↑ — 둘 다 '관심도'가 원인), 인사(교육 많이 받은 직원이 성과↑ — '원래 우수한 사람'이 교란), 의료(특정 약 복용자가 더 건강 — '건강 관리 습관'이 교란) 등 교란은 어디에나 있습니다.

## 🚦 짚고 넘어가기 — Part 4 상관 ≠ 인과

1. 교란변수(Confounder)란 무엇이고, 어떻게 허위상관을 만드나요?
2. "상관이 인과가 아닌" 세 가지 경우를 말해보세요.
3. 교란을 다루는(통제하는) 두 가지 방법은 무엇인가요?

> 💡 **다음 Part 예고:** 이제 관계를 *식*으로 표현하는 회귀로 들어갑니다. 가장 단순한 형태 — 변수 하나로 결과를 설명하는 **단순 회귀**부터 Part 5에서 시작합니다.

# Part 5. 단순 회귀 — 한 변수로 결과를 설명하기

상관계수는 "관계의 세기"를 한 숫자로 줬습니다. 회귀는 한 걸음 더 나아가 그 관계를 **식(직선)**으로 표현합니다. 식이 있으면 *예측*도 하고, "x가 1 늘면 y가 얼마 변하는지"도 말할 수 있습니다.

> ❓ **이 파트에서 답할 질문:** 한 변수와 결과의 관계를 직선 식으로 어떻게 추정하고, 그 기울기와 $R^2$를 어떻게 해석할까요?

## 💡 쉽게 말하면 — 점들을 가장 잘 지나는 직선 긋기

산점도 위에 점들을 가장 잘 통과하는 직선 하나를 긋는다고 상상해보세요. "가장 잘 통과한다"는 건, 각 점에서 직선까지의 *세로 거리(오차)*를 제곱해서 모두 더한 값이 *가장 작아지는* 직선이라는 뜻입니다. 이 방법을 **최소제곱법(OLS, Ordinary Least Squares)**이라고 해요. 앞서 분산을 구할 때처럼, 여기서도 오차를 제곱해서 다룹니다(부호 상쇄를 막고 큰 오차에 더 큰 벌점을 주려고 합니다).

## 회귀식과 계수의 의미

단순 회귀의 식은 이렇습니다.

$$\hat{y} = \beta_0 + \beta_1 x$$

- $\beta_0$ (절편): $x=0$일 때 예측값
- $\beta_1$ (기울기): **$x$가 1 단위 늘 때 $\hat{y}$가 평균적으로 변하는 양** ← 회귀의 핵심 해석
- $R^2$ (결정계수): $y$의 전체 변동 중 이 회귀선이 *설명한* 비율(0~1). 1에 가까울수록 잘 설명.

<img src="https://upload.wikimedia.org/wikipedia/commons/3/3a/Linear_regression.svg" width="560" alt="Linear regression.svg">

최소제곱 회귀선. 파란 점이 관측값, 빨간 선이 적합된 직선이고, 점에서 선까지의 **수직 거리(잔차)** 를 제곱해 더한 값이 최소가 되도록 선을 고릅니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Linear regression.svg` (Public domain) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Linear_regression.svg

### 📖 용어 세 개만 확실히

| 용어 | 기호 | 뜻 |
| --- | --- | --- |
| **적합값(Fitted value)** | $\hat{y}_i$ | 회귀식이 예측한 값 |
| **잔차(Residual)** | $e_i = y_i - \hat{y}_i$ | 실제값 − 예측값. *우리가 계산할 수 있는* 오차 |
| **오차(Error)** | $\varepsilon_i$ | 참인 모형에서의 어긋남. *관측할 수 없는* 이론적 개념 |

> 💡 잔차는 오차의 *추정치*입니다. 그래서 진단(Part 7)에서 우리는 오차의 성질(등분산·정규성)을 **잔차로** 점검합니다.

> $\beta_1$의 부호는 상관계수의 부호와 같습니다. 다만 상관은 단위가 없는 $-1$~$1$ 값이고, 회귀계수는 *실제 단위*(x 한 단위당 y의 변화량)를 가진다는 게 큰 차이입니다 — 그래서 회귀가 더 많은 이야기를 해줍니다.

## 예제 · BMI로 질병 진행도 설명하기 (단순 회귀)

In [ ]:
# [C9] 단순 회귀 — BMI로 질병 진행도 설명 (statsmodels OLS)
X = sm.add_constant(df["bmi"])      # 절편(beta0)을 위해 상수항 추가
y = df["target"]
model = sm.OLS(y, X).fit()                                        # 최소제곱 회귀 적합
print(model.summary().tables[1])     # 계수 표
print(f"\nR² = {model.rsquared:.3f}   (질병 진행도 변동의 약 {model.rsquared*100:.0f}%를 BMI가 설명)")
print(f"기울기 β1 = {model.params['bmi']:.1f}  → BMI가 1(표준화 단위) 늘면 진행도가 평균 {model.params['bmi']:.0f} 증가")

> **읽는 법:** 계수 표의 `coef`가 추정된 $\beta$, `P>|t|`가 그 계수의 p값(0에 가까우면 유의), `[0.025 0.975]`가 95% 신뢰구간입니다. BMI의 계수가 양수이고 p가 매우 작으니 "BMI는 질병 진행도와 유의한 양의 관계"입니다. $R^2$는 BMI 하나로 설명되는 부분이라 그리 높지 않은데 — 그래서 변수를 더 넣는 *다중 회귀*가 필요해집니다.

### 단위가 있는 데이터로 한 번 더 — 엥겔의 법칙 (`engel`)

`diabetes`의 BMI는 미리 표준화된 값이라 "1 늘면"이 무슨 뜻인지 감이 잘 안 옵니다. 그래서 **단위가 살아 있는 데이터**로 같은 회귀를 해봅니다.

`engel`은 19세기 벨기에 가계 235곳의 **연간 소득**과 **식료품 지출**입니다(단위: 벨기에 프랑). 경제학자 엥겔이 발견한 유명한 규칙 — *소득이 늘면 식료품 지출도 늘지만, 소득에서 식료품이 차지하는 **비중**은 줄어든다* — 을 확인할 수 있습니다.

## 예제 · 단위가 있는 회귀 — 엥겔의 법칙

In [ ]:
# [C9b] 단순 회귀 — 소득으로 식료품 지출 설명 (단위가 있는 실데이터)
Xe = sm.add_constant(engel["income"])                             # 절편(β0)을 위해 상수항 1 열을 추가
me = sm.OLS(engel["foodexp"], Xe).fit()                           # 최소제곱 회귀 적합
print(me.summary().tables[1])                                     # 회귀 결과에서 '계수 표'만 출력

b0, b1 = me.params["const"], me.params["income"]
print(f"\n회귀식: 식료품지출 = {b0:.1f} + {b1:.3f} × 소득")
print(f"해석  : 소득이 **100프랑** 늘면 식료품 지출은 평균 **{b1*100:.1f}프랑** 늘어난다")
print(f"        (소득이 0일 때의 예측 지출 = {b0:.0f}프랑 — 기초 식비에 해당하는 절편)")
print(f"R² = {me.rsquared:.3f}  → 식료품 지출 변동의 약 {me.rsquared*100:.0f}%를 소득이 설명")

# 엥겔의 법칙: 소득이 늘수록 '식료품 비중'은 줄어드는가?
share_low  = (engel.loc[engel["income"] <= engel["income"].median(), "foodexp"]
              / engel.loc[engel["income"] <= engel["income"].median(), "income"]).mean()
share_high = (engel.loc[engel["income"] >  engel["income"].median(), "foodexp"]
              / engel.loc[engel["income"] >  engel["income"].median(), "income"]).mean()
print(f"\n[엥겔의 법칙 확인] 식료품 지출이 소득에서 차지하는 비중")
print(f"  소득 하위 절반: {share_low:.1%}   소득 상위 절반: {share_high:.1%}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))                        # 그래프 하나를 그릴 도화지 준비
ax.scatter(engel["income"], engel["foodexp"], alpha=0.5, s=22, label="가계 235곳")
xs = np.linspace(engel["income"].min(), engel["income"].max(), 100)  # 구간을 균등 간격으로 쪼갠 x축 값
ax.plot(xs, b0 + b1 * xs, color="indianred", lw=2, label=f"OLS: y = {b0:.0f} + {b1:.2f}x")
ax.set_xlabel("annual income (Belgian francs)"); ax.set_ylabel("food expenditure")
ax.set_title("Engel curve: income vs food expenditure"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법:** 기울기 0.485는 **"소득이 1프랑 늘면 식료품 지출이 평균 0.49프랑 늘어난다"** 는 뜻입니다. 표준화된 계수와 달리 곧바로 말이 되죠 — 이것이 단위가 있는 데이터로 회귀를 배우는 이유입니다. 절편 147은 "소득이 0이어도 대략 이만큼은 먹는다"로 읽히지만, *데이터에 소득 0인 가계가 없으므로* 절편의 해석은 늘 조심해야 합니다(**외삽의 위험**).
>
> 비중 계산에서 엥겔의 법칙도 확인됩니다 — 식료품이 소득에서 차지하는 비중이 소득 하위 절반 약 68.6%에서 상위 절반 약 62.5%로 낮아집니다(지출액 자체는 늘지만 *비중*은 줄어듦). 기울기(0.49)가 평균 비중(약 0.6)보다 작다는 사실 자체가 이미 그 법칙을 함축합니다.
>
> 💡 **그림에서 하나 더 보이는 것:** 오른쪽으로 갈수록(소득이 높을수록) 점들이 **위아래로 더 넓게 퍼집니다.** 잔차의 퍼짐이 일정하지 않다는 신호 — **이분산(Heteroscedasticity)** 입니다. Part 7에서 이 데이터로 이분산을 정식으로 진단합니다.

## 그림으로 · 회귀선과 잔차

In [ ]:
# [C10] 회귀선과 데이터, 그리고 '오차(잔차)'를 함께 그려보기
fig, ax = plt.subplots(figsize=(7.5, 4.8))                        # 그래프 하나를 그릴 도화지 준비
ax.scatter(df["bmi"], y, alpha=0.4, s=20, label="data")
xs = np.linspace(df["bmi"].min(), df["bmi"].max(), 100)           # 구간을 균등 간격으로 쪼갠 x축 값
ax.plot(xs, model.params["const"] + model.params["bmi"]*xs, color="indianred", lw=2, label="OLS fit")
# 잔차(점에서 선까지의 세로 거리) 몇 개를 표시
for i in range(0, len(df), 30):
    xi, yi = df["bmi"].iloc[i], y.iloc[i]
    yhat = model.params["const"] + model.params["bmi"]*xi
    ax.plot([xi, xi], [yi, yhat], color="gray", lw=0.8, alpha=0.7)
ax.set_xlabel("BMI"); ax.set_ylabel("disease progression"); ax.legend()
ax.set_title("Simple regression: line that minimizes squared residuals")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 빨간 직선이 회귀선이고, 회색 세로선이 각 점의 **잔차(residual = 실제값 − 예측값)**입니다. OLS는 이 회색 선들의 *길이를 제곱해 합한 값*을 가장 작게 만드는 직선을 찾습니다. 잔차가 작을수록(점이 선에 가까울수록) 회귀가 잘 맞는 겁니다.

## 회귀로 예측해보기 — 그리고 R²에 대한 흔한 오해

회귀식이 있으면 *새로운 x*에 대한 예측값을 뽑을 수 있습니다. 다만 예측에는 두 종류의 '불확실성 구간'이 있습니다. **(1) 평균 신뢰구간(Mean CI)**은 "이 x를 가진 사람들의 *평균* 결과가 어디쯤일까"이고, **(2) 예측구간(Prediction Interval)**은 "이 x를 가진 *한 개인*의 결과가 어디쯤일까"입니다. 개인의 변동까지 품어야 하므로 예측구간이 항상 더 넓습니다.

In [ ]:
# [C11] 새 입력에 대한 예측 — 평균 신뢰구간 vs 예측구간
Xs = sm.add_constant(df["bmi"]); ms = sm.OLS(df["target"], Xs).fit()  # 절편(β0)을 위해 상수항 1 열을 추가
q = df["bmi"].quantile([0.25, 0.5, 0.75]).values                  # 분위수 — 정렬했을 때 몇 % 지점의 값
new = pd.DataFrame({"const": [1.0]*3, "bmi": q})
pr = ms.get_prediction(new).summary_frame(alpha=0.05)
out = pd.DataFrame({
    "bmi": q.round(3),
    "예측 ŷ": pr["mean"].round(1),
    "평균 95%CI": [f"[{l:.0f}, {u:.0f}]" for l, u in zip(pr["mean_ci_lower"], pr["mean_ci_upper"])],
    "예측구간": [f"[{l:.0f}, {u:.0f}]" for l, u in zip(pr["obs_ci_lower"], pr["obs_ci_upper"])],
})
print(out.to_string(index=False))
print("\n→ 예측구간(개인)이 평균 신뢰구간보다 훨씬 넓은 것을 확인하세요.")

> **읽는 법:** BMI가 클수록 예측값(ŷ)이 커집니다. 같은 x에서도 *평균* 구간은 좁지만 *개인* 예측구간은 넓습니다 — "평균적으로는 이쯤이지만, 한 사람은 훨씬 다를 수 있다"는 현실을 반영합니다.

이제 **$R^2$에 대한 흔한 오해** 셋을 짚어두겠습니다. ① **"$R^2$가 높으면 좋은 모델"이 아닙니다.** 예측 정확도가 목적이면 $R^2$가 중요하지만, *관계의 존재*를 검정하는 게 목적이면 작은 $R^2$라도 계수가 유의하고 의미 있을 수 있습니다(사람 행동·의학 데이터는 $R^2$가 원래 낮습니다). ② **$R^2$가 높다고 인과가 입증되는 것도 아닙니다** — 잘 맞는 회귀선과 "원인"은 별개입니다. ③ **$R^2$는 변수를 더할수록 무조건 오릅니다** — 그래서 의미 없는 변수까지 넣으면 과적합되니, 변수 수를 보정한 *조정 $R^2$*로 모델을 비교합니다.

## ⌨️ 백문이 불여일타! (2) — s5로 target 단순 회귀

```
[문제]
1) s5(혈청 지표)로 질병 진행도(target)를 설명하는 단순 회귀를 적합해보세요.
2) 기울기와 R²를 BMI 모델과 비교해보세요.
```

In [ ]:
# [C12] ⌨️ 백문이 불여일타! (2) — s5로 target 단순 회귀 적합·비교

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C9]에서 `bmi`로 적합한 코드와 구조가 같습니다 — 설명변수만 `s5`로 바꾸면 됩니다.
- 순서를 기억하세요: `sm.add_constant()`로 상수항을 붙인 뒤 `sm.OLS(y, X).fit()`입니다.
- 기울기는 `params`, 결정계수는 `rsquared` 속성에서 꺼낼 수 있습니다.

</details>

## 🚦 짚고 넘어가기 — Part 5 단순 회귀

1. OLS(최소제곱법)는 어떤 직선을 "가장 좋은" 직선으로 고르나요?
2. 회귀 기울기 $\beta_1$을 말로 풀면 어떤 뜻인가요?
3. $R^2$가 낮다면 무엇을 시도해볼 수 있을까요?

> 💡 **다음 Part 예고:** 변수 하나로는 부족합니다. 여러 변수를 *동시에* 넣으면 무슨 일이 벌어질까요? **다중 회귀**와 "통제된 효과"의 개념을 Part 6에서 만납니다.

# Part 6. 다중 회귀 — 여러 변수를 한꺼번에

현실에서 결과는 한 가지 원인으로 정해지지 않습니다. 질병 진행도는 BMI만이 아니라 혈압·혈액 지표 등 여러 요인이 함께 빚어냅니다. **다중 회귀(Multiple Regression)**는 여러 설명변수를 동시에 넣어, 각 변수의 *고유한 기여*를 분리해냅니다.

> ❓ **이 파트에서 답할 질문:** 여러 변수를 함께 넣으면 각 계수의 의미가 어떻게 달라지고, 왜 그것이 '통제된 효과'일까요?

## 💡 쉽게 말하면 — "다른 조건이 같을 때"

다중 회귀의 마법 같은 점은 이겁니다. 어떤 변수의 계수는 **"나머지 변수를 모두 고정(통제)했을 때, 이 변수만 1 늘면 결과가 얼마 변하는가"**를 뜻합니다. 영어로 *"holding others constant"* 또는 *"다른 조건이 같다면(ceteris paribus)"*이라고 합니다.

예를 들어 "혈압의 계수"는 *BMI·나이 등이 같은 사람들끼리 비교했을 때* 혈압이 미치는 순수한 몫입니다. 바로 이 '통제' 덕분에, Part 4에서 본 교란을 어느 정도 걷어낼 수 있습니다. (단, *측정해서 모델에 넣은* 교란만 통제됩니다 — 빠뜨린 교란은 여전히 위험합니다.)

## 회귀식의 확장

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_k x_k$$

- 각 $\beta_j$ = *다른 변수를 통제한* $x_j$의 평균 효과
- **F-검정:** "모든 계수가 0이다(이 회귀는 쓸모없다)"는 귀무가설을 한 번에 검정 — ANOVA의 그 F입니다
- **조정 $R^2$(adjusted):** 변수를 늘리면 $R^2$는 무조건 오르므로, 변수 수에 벌점을 준 보정 버전. 모델 비교에 씁니다

## 예제 · 여러 지표로 질병 진행도 설명 (다중 회귀)

In [ ]:
# [C13] 다중 회귀 — 여러 지표로 질병 진행도 설명
features = ["age", "sex", "bmi", "bp", "s1", "s3", "s5"]
X = sm.add_constant(df[features])                                 # 절편(β0)을 위해 상수항 1 열을 추가
y = df["target"]
mfit = sm.OLS(y, X).fit()                                         # 최소제곱 회귀 적합
print(mfit.summary().tables[1])                                   # 회귀 결과에서 '계수 표'만 출력
print(f"\nR² = {mfit.rsquared:.3f},  조정 R² = {mfit.rsquared_adj:.3f}")  # 조정 R² — 변수 수에 벌점을 준 값
print(f"F-검정: F = {mfit.fvalue:.1f}, p = {mfit.f_pvalue:.2e}  → 모델 전체가 유의")

> **읽는 법:** 단순 회귀(BMI만)보다 $R^2$가 올랐습니다 — 여러 변수가 함께 더 많은 변동을 설명합니다. 각 계수의 `P>|t|`를 보면 *다른 변수를 통제한 뒤에도* 유의한 변수(bmi·bp·s5 등)와 그렇지 않은 변수가 갈립니다. F-검정 p가 0에 가까우니 "이 회귀가 통째로 쓸모없다"는 가설은 기각됩니다.

## 그림으로 · 계수와 95% 신뢰구간 (forest plot)

In [ ]:
# [C14] 계수와 95% 신뢰구간을 한눈에 — forest plot (개념 도식: matplotlib 직접 구성)
params = mfit.params.drop("const")
ci = mfit.conf_int().drop("const")                                # 계수의 95% 신뢰구간
fig, ax = plt.subplots(figsize=(8, 4.5))                          # 그래프 하나를 그릴 도화지 준비
yloc = np.arange(len(params))                                     # 정수 눈금 배열 생성
ax.errorbar(params.values, yloc,                                  # 오차막대(신뢰구간) 표시
            xerr=[params.values - ci[0].values, ci[1].values - params.values],
            fmt="o", color="steelblue", capsize=4)
ax.axvline(0, color="indianred", ls="--", lw=1)     # 0 = 효과 없음
ax.set_yticks(yloc); ax.set_yticklabels(params.index)
ax.set_xlabel("coefficient (95% CI)"); ax.set_title("Multiple regression coefficients")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 각 점이 계수 추정치, 가로선이 95% 신뢰구간입니다. 구간이 **빨간 0선을 가로지르면** "그 변수의 효과가 0일 가능성을 배제 못 함"(유의하지 않음)이고, 0선 한쪽에 떨어져 있으면 유의합니다. 앞서 배운 신뢰구간 읽기가 회귀에서도 똑같이 작동합니다.

> 📌 **다른 산업에서는?** 다중 회귀는 마케팅 믹스 모델링(여러 채널의 매출 기여 분리), 부동산 가격 모델(면적·위치·층 통제), 인사 분석(성과 요인 분해), 헬스케어(위험 요인 보정)에 광범위하게 쓰입니다.

### "통제한다"는 말의 의미 — 실데이터로 체감하기 (`CASchools`)

다중 회귀의 계수가 "다른 변수를 통제한 효과"라는 말이 아직 추상적일 수 있습니다. 계수가 **극적으로 바뀌는** 실제 사례로 체감해봅시다.

캘리포니아 학군 420곳의 데이터로 묻습니다 — **"학생-교사비(교사 1명당 학생 수)가 낮으면 시험 점수가 높을까?"** 먼저 단순 회귀로 보고, 그다음 학생 배경 변수들을 함께 넣어 봅니다.

| 변수 | 뜻 |
| --- | --- |
| `testscr` | 학군 평균 시험 점수(읽기·수학 평균) — **결과** |
| `str_ratio` | 학생-교사비(학생 수 ÷ 교사 수) — **관심 변수** |
| `english` | 영어 학습자 비율(%) — 학생 배경 |
| `lunch` | 무상급식 대상 비율(%) — 빈곤 지표 |
| `income` | 지역 평균 소득 |

## 예제 · '통제'의 위력 — CASchools 단순 vs 다중

In [ ]:
# [C14b] 통제의 위력 — 단순 회귀 vs 다중 회귀에서 같은 계수가 어떻게 달라지나
simple = smf.ols("testscr ~ str_ratio", data=ca).fit()
multi  = smf.ols("testscr ~ str_ratio + english + lunch + income", data=ca).fit()

print("[단순 회귀] testscr ~ str_ratio")
print(simple.summary().tables[1])                                 # 회귀 결과에서 '계수 표'만 출력
print(f"R² = {simple.rsquared:.3f}")

print("\n[다중 회귀] + english + lunch + income (학생 배경 통제)")
print(multi.summary().tables[1])                                  # 회귀 결과에서 '계수 표'만 출력
print(f"R² = {multi.rsquared:.3f},  조정 R² = {multi.rsquared_adj:.3f}")  # 조정 R² — 변수 수에 벌점을 준 값

b_s, b_m = simple.params["str_ratio"], multi.params["str_ratio"]
print(f"\n학생-교사비 계수:  단순 {b_s:.2f}  →  통제 후 {b_m:.2f}"
      f"   ({(1 - b_m/b_s):.0%} 축소)")
print("해석: 학생 1명당 교사를 줄일 때의 '점수 하락'이, 학생 배경을 통제하면 훨씬 작아집니다.")

> **읽는 법:** 단순 회귀에서 학생-교사비 계수는 약 **−2.28**입니다("학생-교사비가 1 늘면 점수가 2.28점 낮다"). 그런데 학생 배경(영어 학습자·무상급식·소득)을 통제하면 계수가 약 **−0.56**으로 **1/4 수준으로 줄어듭니다.**
>
> 왜 그럴까요? *학급이 큰 학군은 대체로 저소득·영어 학습자 비율이 높은 지역*이었습니다. 단순 회귀의 −2.28에는 **"학급 크기의 효과"와 "학생 배경의 효과"가 섞여** 있었던 것입니다. 이것이 바로 Part 4에서 배운 **교란**이고, 다중 회귀에 교란변수를 넣는 것이 **통제**입니다.
>
> 💡 이 사례가 주는 실무 감각 두 가지:
> 1. **계수는 어떤 변수와 함께 있느냐에 따라 달라진다.** "학생-교사비의 효과"라는 단일한 숫자는 없고, 항상 "무엇을 통제했을 때의 효과"입니다. 그래서 보고서에는 *모형에 넣은 변수 목록*을 반드시 함께 적습니다.
> 2. **그래도 남은 −0.56이 '진짜 인과'라는 보장은 없습니다.** 우리가 *측정해서 넣은* 교란만 통제됐을 뿐, 학부모의 교육열·교사의 역량처럼 측정하지 못한 것은 그대로 남습니다(누락변수 편향). 이 한계를 정면으로 다루는 것이 D016·D017입니다.
>
> ⚠️ **반대 방향의 함정도 있습니다.** 통제를 많이 하는 것이 늘 좋은 것은 아닙니다. *결과의 원인이 아니라 결과가 만든 변수*(사후 변수)나 *중간 경로에 있는 변수(매개변수)* 를 통제하면 오히려 편향이 생깁니다. 이를 **나쁜 통제(bad control)** 라고 하며, D017에서 다시 만납니다.

## 어느 변수가 더 중요할까 — 표준화 계수로 보는 상대 중요도

다중 회귀에서 자연스럽게 떠오르는 질문 — "그래서 *어느 변수가 가장 중요*한가?"입니다. 그런데 계수의 크기를 그냥 비교하면 함정이 있습니다. 변수마다 단위(스케일)가 다르면, 계수 크기는 *중요도*가 아니라 *단위*를 반영할 뿐이거든요. 예컨대 '연봉(원)'의 계수와 '나이(세)'의 계수를 숫자만으로 비교하는 건 무의미합니다.

해법은 **변수를 표준화(각자 평균 0, 표준편차 1로)한 뒤 회귀**하는 겁니다. 그러면 모든 계수가 "그 변수가 *1 표준편차* 늘 때 결과가 몇 변하나"라는 *같은 잣대*가 돼, 크기를 직접 비교할 수 있습니다. 다행히 diabetes 데이터는 이미 표준화돼 있어서, 계수의 절댓값을 *상대 중요도*로 바로 읽을 수 있습니다.

In [ ]:
# [C15] 표준화 계수의 절댓값 = 상대적 기여도 (diabetes는 이미 표준화돼 있음)
imp = mfit.params.drop("const").abs().sort_values()
fig, ax = plt.subplots(figsize=(8, 4))                            # 그래프 하나를 그릴 도화지 준비
ax.barh(imp.index, imp.values, color="steelblue")                 # 가로 막대그래프
ax.set_xlabel("|coefficient|  (standardized → 상대 중요도)")
ax.set_title("Relative importance of predictors")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print("가장 크게 기여하는 변수:", imp.index[-1], f"(|계수| {imp.values[-1]:.0f})")

> **읽는 법(그림):** 막대가 길수록 *다른 변수를 통제한 뒤에도* 결과에 크게 기여하는 변수입니다. bmi·s5 같은 변수가 위쪽에 옵니다. 단, 이건 "이 표본·이 모델 안에서의 상대적 기여"일 뿐, *인과적 중요도*가 아님을 기억하세요(빠진 교란·다중공선성이 이 순위를 흔들 수 있습니다).

> ➕ **더 알고 싶다면:** 표준화 계수 외에도 변수 중요도를 보는 방법은 여러 가지입니다(부분 $R^2$, 순열 중요도 등). 방법마다 "중요하다"의 정의가 조금씩 달라서, 하나의 숫자에 절대적 의미를 부여하기보다 *여러 각도로* 보는 게 안전합니다.

## ⌨️ 백문이 불여일타! (3) — 단순 vs 다중 회귀의 bmi 계수 비교

```
[문제]
1) 위 다중 회귀의 bmi 계수와 Part 5 단순 회귀의 bmi 계수를 나란히 출력해 비교해보세요.
2) 값이 달라졌다면 왜 그런지 "통제" 개념으로 설명해보세요.
```

In [ ]:
# [C16] ⌨️ 백문이 불여일타! (3) — 단순 vs 다중 회귀의 bmi 계수 비교

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- 단순 회귀 쪽은 [C9]처럼 `bmi` 하나만 넣어 다시 적합한 뒤 `params["bmi"]`를 꺼내면 됩니다.
- 다중 회귀 쪽은 이미 적합해 둔 `mfit.params["bmi"]`에서 바로 가져올 수 있습니다.
- 두 값을 나란히 출력하고, "다른 변수를 통제했는가"의 차이로 설명해보세요.

</details>

## 🚦 짚고 넘어가기 — Part 6 다중 회귀

1. 다중 회귀에서 한 변수의 계수는 "무엇을 통제한" 효과인가요?
2. F-검정과 개별 계수의 t-검정은 각각 무엇을 검정하나요?
3. 조정 $R^2$가 일반 $R^2$와 다른 이유는 무엇인가요?

> 💡 **다음 Part 예고:** 회귀 결과가 나왔다고 끝이 아닙니다. "이 회귀를 *믿어도 되나*?"를 따지는 **진단**이 남았습니다. 다중공선성·잔차·이상치를 Part 7에서 점검합니다.

# Part 7. 회귀 진단 — 이 회귀, 믿어도 되나

회귀계수와 $R^2$가 나왔다고 곧장 믿으면 위험합니다. 회귀에는 몇 가지 *전제*가 있고, 그게 깨지면 계수와 p값이 엉뚱해질 수 있습니다. 그래서 **진단(Diagnostics)**으로 모델의 건강 상태를 점검합니다.

진단을 건너뛰면 어떤 일이 벌어질까요? 회귀는 *어떤 데이터를 넣어도* 계수와 p값을 군말 없이 뱉어냅니다 — 가정이 심각하게 깨진 경우에도요. 그래서 진단 없이 결과를 그대로 믿으면, 멀쩡해 보이는 숫자(작은 p, 그럴듯한 계수)에 속아 잘못된 결론을 오히려 *확신*하게 됩니다. 가장 위험한 상태입니다. 비유하자면 진단은 *비행 전 점검*과 같습니다. 엔진에 시동이 걸린다고(=회귀가 결과를 낸다고) 곧장 이륙하지 않고, 계기와 부품을 하나씩 확인하는 일입니다. 그 점검을 마쳐야 "이 모델을 어디까지, 어떤 조건에서 믿을 수 있는지"를 자신 있게 말할 수 있습니다. 다행히 점검 항목은 정해져 있고(바로 아래 다섯 가지 가정), 파이썬이 숫자와 그림으로 점검을 도와줍니다 — 생각보다 어렵지 않으니 하나씩 따라와 보세요.

> ❓ **이 파트에서 답할 질문:** 회귀를 신뢰하기 전에 무엇을(다중공선성·잔차·이상치) 어떻게 점검해야 할까요?

## 잠깐 정리 — 회귀가 기대는 다섯 가지 가정

회귀의 계수와 p값을 믿으려면 몇 가지 전제가 충족돼야 합니다. 진단으로 점검할 것들을 한자리에 모아두겠습니다.

1. **선형성(Linearity):** x와 y의 관계가 (변환 후라도) 직선이다. → 잔차 vs 적합값에 곡선 패턴이 없어야 합니다.
2. **독립성(Independence):** 관측치들이 서로 독립이다. → 시계열·군집 데이터면 특별한 처리가 필요합니다.
3. **등분산성(Homoscedasticity):** 잔차의 퍼짐이 일정하다. → 나팔 모양이면 robust 표준오차로 대응합니다.
4. **정규성(Normality):** *잔차*가 대략 정규분포를 따른다. → Q-Q plot으로 확인(표본이 크면 덜 민감).
5. **다중공선성 없음:** 설명변수끼리 과하게 얽히지 않는다. → VIF로 점검합니다.

> 💡 **중요한 균형 감각:** 이 가정들이 *완벽히* 충족되는 실제 데이터는 거의 없습니다. 핵심은 "위반이 있는가"가 아니라 **"위반이 결론을 바꿀 만큼 심각한가"**예요. 작은 위반은 대체로 견딜 만하고, 심각한 위반은 변수 변환·robust 기법·다른 모델로 대응합니다. 진단은 *모델을 버리려고*가 아니라 *얼마나 믿을지 정하려고* 하는 일입니다. 아래에서 ①다중공선성 → ②잔차 → ③영향점 → ④등분산 순으로 하나씩 점검해봅니다.

## 진단 ① 다중공선성 — 설명변수끼리 너무 닮았을 때

설명변수들끼리 강하게 상관하면(예: '키(cm)'와 '키(inch)'를 둘 다 넣기), 회귀가 "누구 덕분에 설명됐는지" 헷갈려합니다. 그 결과 **계수가 불안정해지고(부호가 뒤집히거나 표준오차가 폭발)** 해석이 어려워집니다. 이걸 **다중공선성(Multicollinearity)**이라 합니다.

진단 지표는 **VIF(분산팽창계수, Variance Inflation Factor)**입니다.

- VIF = 1 : 다른 변수와 독립적 (이상적)
- VIF 5~10 이상 : 다중공선성 의심 → 변수 제거·결합·정규화 고려

> 💡 longley 데이터는 거시경제 변수들(GNP·인구·연도 등)이 모두 시간에 따라 함께 증가해, **다중공선성의 교과서 사례**입니다.

In [ ]:
# [C17] VIF 계산 — longley 데이터 (다중공선성 교과서)
Xl = sm.add_constant(longley[["GNP", "GNPDEFL", "UNEMP", "ARMED", "POP", "YEAR"]])  # 절편(β0)을 위해 상수항 1 열을 추가
vif = pd.DataFrame({
    "변수": Xl.columns,
    "VIF": [variance_inflation_factor(Xl.values, i) for i in range(Xl.shape[1])]  # VIF — 5~10 넘으면 다중공선성 의심
})
print(vif.to_string(index=False))
print("\n→ GNP·POP·YEAR·GNPDEFL의 VIF가 수백~수천! 서로 거의 같은 정보(시간 추세)를 담고 있습니다.")

> **읽는 법:** `const`(상수항)의 VIF는 무시하고, 설명변수들을 봅니다. GNP·POP·YEAR 등이 어마어마한 VIF를 보입니다 — 이들은 사실상 "같은 추세"를 중복해서 담고 있습니다. 이런 모델의 개별 계수는 *믿기 어렵습니다*. 변수 중 일부만 남기거나 결합해야 해요.

## 진단 ② 잔차 — 회귀가 놓친 패턴이 있나

**잔차(residual = 실제값 − 예측값)**에는 회귀가 *설명하지 못한 나머지*가 담깁니다. 회귀가 건강하다면 잔차는 *아무 패턴 없이* 0 주위에 고르게 흩어져 있어야 해요(등분산·독립·정규). 잔차에 곡선·깔때기 모양이 보이면, 모델이 뭔가 놓치고 있다는 신호입니다.

## 진단 ② 예제 · 잔차 vs 적합값과 Q-Q plot

In [ ]:
# [C18] 잔차 진단 — diabetes 다중회귀의 잔차 두 그림
resid = mfit.resid; fitted = mfit.fittedvalues                    # 회귀가 예측한 값(적합값)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))                   # 그림 한 장에 그래프 2개(1행 2열)
# (1) 잔차 vs 적합값 — 패턴/깔때기가 없어야 건강
axes[0].scatter(fitted, resid, alpha=0.4, s=18)
axes[0].axhline(0, color="indianred", ls="--")                    # 가로 기준선 표시
axes[0].set_xlabel("fitted (예측값)"); axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs Fitted (패턴 없어야 좋음)")
# (2) 잔차 Q-Q plot — 직선에 붙어야 정규
stats.probplot(resid, dist="norm", plot=axes[1])                  # Q-Q plot — 점이 직선에 붙을수록 그 분포에 가깝다
axes[1].set_title("Q-Q plot of residuals (정규성)")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 왼쪽은 잔차가 0선 주위에 *구름처럼 무패턴*으로 퍼져야 건강합니다(곡선이나 나팔 모양이면 비선형·이분산 신호). 오른쪽 Q-Q plot은 점들이 직선에 붙을수록 잔차가 정규에 가깝다는 뜻입니다. 큰 위반이 없으면 회귀의 p값·신뢰구간을 신뢰할 수 있습니다.

## 진단 ③ · 이상치 영향 (Cook's distance)

In [ ]:
# [C19] 진단 ③ 이상치 영향 — Cook's distance (한 점이 회귀선을 흔드나)
infl = mfit.get_influence()                                       # 영향점 진단용 객체 (Cook's distance 등)
cooks = infl.cooks_distance[0]
fig, ax = plt.subplots(figsize=(9, 3.5))                          # 그래프 하나를 그릴 도화지 준비
ax.stem(np.arange(len(cooks)), cooks, markerfmt=" ", basefmt=" ")  # 관측치별 값을 막대로 — 튀는 점 찾기
thresh = 4 / len(cooks)        # 흔한 경험적 임계값
ax.axhline(thresh, color="indianred", ls="--", label=f"threshold ≈ {thresh:.3f}")  # 가로 기준선 표시
ax.set_xlabel("observation index"); ax.set_ylabel("Cook's distance")
ax.set_title("Influence of each point (높은 막대 = 영향 큰 관측치)"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print(f"임계값 초과 관측치 수: {(cooks > thresh).sum()}개  (이들이 계수를 크게 좌우할 수 있음)")

> **읽는 법(그림):** Cook's distance가 큰(임계선 위로 튀는) 관측치는 *혼자서 회귀선을 끌어당기는* 영향력 있는 점입니다. 이런 점은 입력 오류인지, 특별한 사례인지 확인해야 해요. 무작정 지우면 안 되지만(데이터 조작!), "이 점을 빼면 결론이 바뀌는가"를 점검하는 건 중요한 습관입니다.

> 📌 **다른 산업에서는?** 이상치·영향점 진단은 금융(거래 이상 탐지), 부동산(이례적 거래가 모델 왜곡), 품질관리(불량 한 건이 추세 왜곡) 등에서 모델을 배포하기 전 필수 점검입니다.

## 진단 ④ 등분산 — 잔차의 퍼짐이 일정한가

회귀의 또 다른 전제는 **등분산성(Homoscedasticity)** — 잔차의 퍼짐이 예측값 전 구간에서 *일정*해야 한다는 것입니다. 잔차가 한쪽에서 좁고 다른 쪽에서 넓은 *나팔(깔때기) 모양*이면 **이분산(Heteroscedasticity)** 이라 부르고, 이러면 계수 추정 자체는 괜찮아도 *표준오차·p값·신뢰구간이 틀어집니다.* 매출·소득처럼 값이 클수록 변동도 커지는 데이터에서 흔합니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/1/11/Linear_regression_residuals_heteroscedastic.svg" width="620" alt="Linear regression residuals heteroscedastic.svg">

이분산의 전형적인 모습. x가 커질수록 점들이 회귀선에서 더 멀리 흩어집니다(나팔 모양). 계수(직선의 기울기)는 여전히 쓸 만하지만, 그 **불확실성 추정**이 틀어집니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Linear regression residuals heteroscedastic.svg` (CC0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Linear_regression_residuals_heteroscedastic.svg

눈으로는 '잔차 vs 적합값' 그림에서 나팔 모양을 찾고, 숫자로는 **Breusch-Pagan 검정**으로 확인합니다($H_0$="등분산이다", p가 작으면 이분산 의심).

## 진단 ④ 예제 · Breusch-Pagan 등분산 검정

In [ ]:
# [C20] 등분산 검정 — Breusch-Pagan
from statsmodels.stats.diagnostic import het_breuschpagan
bp_stat, bp_p, _, _ = het_breuschpagan(mfit.resid, mfit.model.exog)  # Breusch-Pagan 등분산 검정 — p가 작으면 이분산
print(f"Breusch-Pagan: LM 통계량 = {bp_stat:.2f}, p = {bp_p:.4f}")
print("H0: 등분산이다 →", "이분산 의심 (p<0.05)" if bp_p < 0.05 else "등분산으로 볼 만함 (p>=0.05)")
print("\n→ 이분산이 의심되면 robust 표준오차(cov_type='HC3')로 다시 적합해 p값을 보정합니다.")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정

> **읽는 법:** `diabetes` 다중 회귀에서 Breusch-Pagan p가 0.05보다 작게 나옵니다 — **등분산 가정이 의심된다**는 뜻입니다. 이럴 때 계수 추정치는 그대로 두고, **이분산에 강한(Robust) 표준오차**로 불확실성만 다시 계산합니다. 바로 아래에서 그 방법과, 이분산이 더 뚜렷한 `engel` 데이터로 효과를 확인합니다.

> 📌 **다른 산업에서는?** 이분산은 금융(고액 거래일수록 변동 큼), 이커머스(대형 고객일수록 매출 변동 큼), 부동산(고가 주택일수록 가격 편차 큼)에서 거의 기본값으로 나타납니다 — robust SE가 실무 표준인 이유입니다.

## 진단 ④ 실사례 · engel 이분산과 robust SE

In [ ]:
# [C20b] 이분산 실사례 — engel(소득→식료품 지출)에서 진단하고 robust SE로 대응
from statsmodels.stats.diagnostic import het_breuschpagan

me = sm.OLS(engel["foodexp"], sm.add_constant(engel["income"])).fit()  # 절편(β0)을 위해 상수항 1 열을 추가
bp_stat_e, bp_p_e, _, _ = het_breuschpagan(me.resid, me.model.exog)  # Breusch-Pagan 등분산 검정 — p가 작으면 이분산
print(f"[engel] Breusch-Pagan: LM = {bp_stat_e:.1f}, p = {bp_p_e:.2e}"
      f"  → {'이분산 (등분산 기각)' if bp_p_e < 0.05 else '등분산으로 볼 만함'}")

# 같은 모형을 robust(HC3) 표준오차로 다시 적합 — 계수는 그대로, SE만 바뀐다
me_rob = sm.OLS(engel["foodexp"], sm.add_constant(engel["income"])).fit(cov_type="HC3")  # 절편(β0)을 위해 상수항 1 열을 추가
comp = pd.DataFrame({
    "계수(coef)":      [me.params["income"],     me_rob.params["income"]],
    "표준오차(SE)":     [me.bse["income"],        me_rob.bse["income"]],
    "t":               [me.tvalues["income"],    me_rob.tvalues["income"]],
    "95% CI 하한":      [me.conf_int().loc["income", 0], me_rob.conf_int().loc["income", 0]],  # 계수의 95% 신뢰구간
    "95% CI 상한":      [me.conf_int().loc["income", 1], me_rob.conf_int().loc["income", 1]],  # 계수의 95% 신뢰구간
}, index=["기본 SE", "robust SE (HC3)"]).round(4)
print("\n[income 계수의 불확실성 — 기본 vs robust]")
print(comp.to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))                          # 그래프 하나를 그릴 도화지 준비
ax.scatter(me.fittedvalues, me.resid, alpha=0.5, s=20)            # 회귀가 예측한 값(적합값)
ax.axhline(0, color="indianred", ls="--")                         # 가로 기준선 표시
ax.set_xlabel("fitted (예측 지출)"); ax.set_ylabel("residual")
ax.set_title("engel: residuals fan out as fitted values grow (heteroscedastic)")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법:** `engel`의 Breusch-Pagan p는 사실상 0 — **이분산이 뚜렷합니다.** 그림에서도 예측 지출이 커질수록 잔차가 부채처럼 넓어집니다(부유한 가계는 식료품 지출의 개인차가 훨씬 큼).
>
> 표를 보면 **계수는 그대로**(0.4852)이고 **표준오차만 커집니다.** 그 결과 신뢰구간이 넓어지고 t는 작아집니다 — *기본 SE를 그대로 믿었다면 우리 추정의 정밀도를 과대평가*했을 겁니다. 이것이 robust 표준오차가 하는 일입니다: **점추정은 손대지 않고, 불확실성만 정직하게 키운다.**
>
> 💡 **실무 규칙:** 금액·매출·소득처럼 값이 클수록 변동이 커지는 변수를 회귀할 때는, 진단 결과를 보기 전에도 `fit(cov_type="HC3")`을 기본으로 두는 분석가가 많습니다. 손해가 거의 없고(등분산이어도 결과가 거의 같음) 이분산일 때 크게 도움이 되기 때문입니다. 또 다른 대응은 **로그 변환**입니다(`np.log(foodexp) ~ np.log(income)`) — 값의 배율 관계를 직선으로 펴면서 이분산도 함께 완화되는 경우가 많습니다.

## ⌨️ 백문이 불여일타! (4) — VIF 최대 변수 제거 후 재계산

```
[문제]
1) longley에서 VIF가 가장 큰 변수 하나를 빼고 다시 VIF를 계산해보세요.
2) 남은 변수들의 VIF가 줄어드는지 확인해보세요.
```

In [ ]:
# [C21] ⌨️ 백문이 불여일타! (4) — VIF 최대 변수 제거 후 재계산

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C17]의 VIF 코드에서 컬럼 리스트만 바꾸면 됩니다 — VIF가 가장 컸던 변수(예: `GNP`)를 빼보세요.
- `sm.add_constant()`로 상수항을 다시 붙인 뒤 `variance_inflation_factor`를 컬럼마다 반복 계산합니다.
- 서로 닮은 변수 중 하나가 빠지면 남은 변수들의 VIF가 어떻게 변하는지 관찰해보세요.

</details>

## 🚦 짚고 넘어가기 — Part 7 회귀 진단

1. 다중공선성은 무엇이고, VIF가 크면 어떤 문제가 생기나요?
2. 건강한 회귀의 잔차는 어떤 모습이어야 하나요?
3. Cook's distance가 큰 관측치를 발견하면 무엇을 해야 하나요(그냥 지우면 될까요)?

> 💡 **다음 Part 예고:** 도구를 다 익혔으니, 회귀가 실제로 *어떻게 쓰이는지* — 예측·검정·인과추정이라는 **세 얼굴**을 Part 8에서 정리합니다.

# Part 8. 회귀의 세 얼굴 — 결국 무엇을 위한 도구인가

같은 회귀 식이라도, 분석가가 무엇을 원하느냐에 따라 *세 가지 다른 얼굴*로 쓰입니다. 이 구분을 알면 "내가 지금 회귀로 무엇을 하려는 건가"가 또렷해집니다.

> ❓ **이 파트에서 답할 질문:** 회귀는 예측·설명·인과 중 무엇을 위한 도구이고, 목적에 따라 무엇을 더 조심해야 할까요?

## 얼굴 ① 예측 (Prediction)

"새 환자의 지표를 넣으면 질병 진행도를 얼마로 예측할까?" — $\hat{y}$를 뽑아내는 용도입니다. 여기서는 *계수의 해석*보다 **예측이 얼마나 정확한가**가 중요합니다. 머신러닝이 회귀를 보는 주된 관점이고, 다중공선성도 (예측만 잘 되면) 크게 문제 삼지 않습니다.

## 얼굴 ② 가설 검정의 일반화 (Inference)

"BMI는 질병 진행도와 *유의한* 관계가 있나?" — 계수의 p값·신뢰구간으로 *관계의 존재*를 검정합니다. 사실 **t-test와 ANOVA는 회귀의 특수한 경우**예요. 집단 구분을 0/1 변수로 넣은 회귀가 곧 t-test이고, 여러 집단을 넣으면 ANOVA가 됩니다. 회귀는 이들을 품는 더 일반적인 틀입니다.

## 얼굴 ③ 인과 효과 추정의 도구 (Causal)

"BMI를 *낮추면* 진행도가 *줄어들까*?" — 교란변수를 함께 넣어 통제함으로써 *인과 효과*에 다가가려는 용도입니다. 가장 야심차지만 가장 위험한 얼굴입니다. **측정해서 넣은 교란만 통제되고, 빠뜨린 교란은 여전히 결과를 오염**시키기 때문입니다.

## 그래서 — "통제 하의 평균 효과"라는 신중한 언어

회귀계수를 해석할 때 가장 안전하고 정확한 표현은 이것입니다.

> *"(모델에 넣은) 다른 변수들이 같은 수준일 때, 이 변수가 1 늘면 결과가 평균적으로 β만큼 다르다(연관된다)."*

"이 변수가 결과를 **일으킨다(Cause)**"가 아니라 "**연관된다(Associated)**"라고 적는 습관 — 이게 데이터를 다루는 사람의 지적 정직함입니다. 관찰 데이터로 회귀를 돌렸다면, 결론은 *연관*까지입니다. **진짜 인과를 말하려면 무작위 실험이나 인과추론 설계가 필요**하고, 그게 바로 다음 시간의 주제입니다.

## 회귀로는 어디까지, 그리고 그 다음은

여기서 한 번 정직하게 짚고 가겠습니다. 다중 회귀의 '통제'는 강력하지만 *만능이 아닙니다.* 우리가 통제할 수 있는 건 **측정해서 모델에 넣은 변수**뿐입니다. 측정하지 못했거나 존재조차 몰랐던 교란변수(예: 환자의 생활습관, 고객의 숨은 동기)는 여전히 계수를 오염시킵니다. 이를 *누락변수 편향(Omitted Variable Bias)*이라고 합니다. 게다가 역인과(결과가 원인에 영향)나 선택편향이 끼면, 변수를 아무리 많이 넣어도 관찰 데이터만으로는 인과를 *증명*할 수 없습니다.

그렇다면 진짜 인과는 어떻게 알까요? 핵심 아이디어는 **"비교 가능한 두 집단을 만든다"**입니다. 처치(Treatment)를 받은 집단과 받지 않은 집단이 *처치 여부 빼고는 똑같다면*, 결과의 차이는 처치 때문이라고 말할 수 있습니다. 이를 가장 깔끔하게 보장하는 방법이 **무작위 배정(Randomization)** — 즉 A/B 테스트입니다. 무작위로 나누면 측정하지 못한 교란까지 *평균적으로* 양쪽에 고루 섞이거든요.

그래서 다음 두 시간은 이렇게 이어집니다. 먼저 무작위 실험(A/B 테스트)과 잠재결과 프레임워크로 "우리가 만든 변화가 진짜 원인인가"를 정면으로 다루고, 이어서 실험을 *할 수 없을 때* 관찰 데이터로 인과에 다가가는 준실험 설계(DiD·PSM 등)를 배웁니다. 오늘 배운 회귀는 그 모든 인과 도구의 *기초 언어*예요 — '통제'라는 발상 자체가 여기서 출발하니까요.

## 예제 · 회귀로 t-test를 그대로 재현하기

In [ ]:
# [C22] 한 얼굴 맛보기 — 회귀로 t-test를 그대로 재현 (얼굴 ②의 증거)
# diabetes의 sex(0/1 비슷한 이분 변수)로 target 차이를 (a) t-test와 (b) 회귀로
grp = (df["sex"] > df["sex"].median()).astype(int)     # 두 그룹으로 이분
g0 = df.loc[grp == 0, "target"]; g1 = df.loc[grp == 1, "target"]
t_res = stats.ttest_ind(g1, g0)                                   # 두 독립 집단의 평균 비교 (equal_var=False면 Welch)
reg = sm.OLS(df["target"], sm.add_constant(grp.rename("group"))).fit()  # 절편(β0)을 위해 상수항 1 열을 추가
print(f"(a) t-test    : t = {t_res.statistic:.3f}, p = {t_res.pvalue:.4f}")
print(f"(b) 회귀 group계수: {reg.params['group']:.1f}, t = {reg.tvalues['group']:.3f}, p = {reg.pvalues['group']:.4f}")
print("→ 회귀의 group 계수 t/p가 t-test와 (거의) 일치! t-test는 회귀의 특수한 경우입니다.")

> **읽는 법:** 두 집단 비교를 0/1 변수 회귀로 돌리면, 그 변수의 계수 검정이 t-test와 같은 t·p를 줍니다. "회귀가 가설검정을 품는다"(얼굴 ②)는 말이 빈말이 아님을 보여줍니다.

## 예제 · 범주형을 넣으면 회귀가 ANOVA가 된다

In [ ]:
# [C23] 한 걸음 더 — 여러 집단(범주형)을 넣으면 회귀가 ANOVA가 된다
df["_grp"] = pd.qcut(df["bmi"], 3, labels=["low", "mid", "high"])     # bmi를 3구간 범주로
groups = [g["target"].values for _, g in df.groupby("_grp", observed=True)]  # 집단별로 묶어서 계산
f_anova, p_anova = stats.f_oneway(*groups)                        # 일원 ANOVA — F = 집단 간 분산 / 집단 내 분산
dummies = pd.get_dummies(df["_grp"], drop_first=True).astype(float)  # 범주형을 0/1 더미 변수로 변환
reg_cat = sm.OLS(df["target"], sm.add_constant(dummies)).fit()    # 절편(β0)을 위해 상수항 1 열을 추가
print(f"(a) 일원 ANOVA   : F = {f_anova:.2f}, p = {p_anova:.2e}")
print(f"(b) 범주형 회귀  : F = {reg_cat.fvalue:.2f}, p = {reg_cat.f_pvalue:.2e}")
print("→ 회귀의 F-검정이 ANOVA와 일치! ANOVA도 회귀의 특수한 경우입니다.")
df.drop(columns="_grp", inplace=True)

> **읽는 법:** 집단(범주형)을 더미 변수로 넣은 회귀의 F-검정이 일원 ANOVA의 F와 같습니다. t-test에 이어 ANOVA까지 — **회귀는 우리가 배운 평균 비교 검정들을 모두 품는 일반 틀**임이 확인됩니다. 회귀를 익히면 통계 검정 대부분을 하나의 관점으로 볼 수 있습니다.

## 🚦 짚고 넘어가기 — Part 8 회귀의 세 얼굴

1. 회귀의 세 얼굴(예측·검정·인과)은 각각 무엇을 목적으로 하나요?
2. 관찰 데이터 회귀 결과를 "인과"로 말하면 안 되는 이유는 무엇인가요?
3. 회귀계수를 해석하는 가장 안전한 문장은 어떻게 쓰나요?

> 💡 **다음 Part 예고:** 배운 걸 모아 실제 데이터에 회귀를 적합하고, 진단하고, *인과 주장 없이* 보고서로 정리하는 종합 실습으로 마무리합니다.

# 🧪 종합 실습 — 회귀 적합·진단·보고

실제 데이터에 다중 회귀를 적합하고, 진단까지 거쳐, **"무엇이 결과에 기여하는가(연관 수준)"**를 보고서로 정리합니다. 핵심 규칙 하나 — **인과 주장 금지.** 관찰 데이터이므로 우리는 *연관*까지만 말합니다.

```text
[1] diabetes 다중회귀 적합 → 계수·R²·F
[2] VIF로 다중공선성 점검 → 신뢰성 평가
[3] 잔차 진단 → 가정 점검
[4] 기여도 보고서 (연관 수준, 인과 주장 없이)
```

## 종합 [1] · diabetes 다중 회귀 적합

In [ ]:
# [C24] [1] diabetes 다중회귀 (핵심 변수)
feat = ["bmi", "bp", "s5", "s3", "age"]
Xf = sm.add_constant(df[feat]); yf = df["target"]                 # 절편(β0)을 위해 상수항 1 열을 추가
fin = sm.OLS(yf, Xf).fit()                                        # 최소제곱 회귀 적합
print("[1] 회귀 계수")
print(fin.summary().tables[1])                                    # 회귀 결과에서 '계수 표'만 출력
print(f"\nR² = {fin.rsquared:.3f}, 조정 R² = {fin.rsquared_adj:.3f}, F p = {fin.f_pvalue:.2e}")  # 조정 R² — 변수 수에 벌점을 준 값

## 종합 [2] · VIF로 다중공선성 점검

In [ ]:
# [C25] [2] VIF 점검  (이 모델의 설명변수끼리 얼마나 얽혔나)
vif_f = pd.DataFrame({"변수": Xf.columns,
    "VIF": [variance_inflation_factor(Xf.values, i) for i in range(Xf.shape[1])]})  # VIF — 5~10 넘으면 다중공선성 의심
print("[2] VIF (const 제외하고 해석)")
print(vif_f.to_string(index=False))

## 종합 [3] · 잔차 진단

In [ ]:
# [C26] [3] 잔차 진단 (가정 점검)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))                 # 그림 한 장에 그래프 2개(1행 2열)
axes[0].scatter(fin.fittedvalues, fin.resid, alpha=0.4, s=16); axes[0].axhline(0, color="indianred", ls="--")  # 가로 기준선 표시
axes[0].set_title("Residuals vs Fitted"); axes[0].set_xlabel("fitted"); axes[0].set_ylabel("residual")
stats.probplot(fin.resid, dist="norm", plot=axes[1]); axes[1].set_title("Q-Q of residuals")  # Q-Q plot — 점이 직선에 붙을수록 그 분포에 가깝다
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print("정규성(잔차) Shapiro p =", round(stats.shapiro(fin.resid).pvalue, 3))  # 정규성 검정 — H0: 정규분포를 따른다

## 종합 [4] · 영향점 (Cook's distance)

In [ ]:
# [C27] [4] 영향점 — Cook's distance로 회귀를 흔드는 관측치 찾기
infl = fin.get_influence(); cooks = infl.cooks_distance[0]        # 영향점 진단용 객체 (Cook's distance 등)
thr = 4 / len(cooks)
fig, ax = plt.subplots(figsize=(9, 3.3))                          # 그래프 하나를 그릴 도화지 준비
ax.stem(np.arange(len(cooks)), cooks, markerfmt=" ", basefmt=" ")  # 관측치별 값을 막대로 — 튀는 점 찾기
ax.axhline(thr, color="indianred", ls="--", label=f"threshold {thr:.3f}")  # 가로 기준선 표시
ax.set_title("Cook's distance (종합 모델)"); ax.set_xlabel("observation"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print(f"영향점(임계 초과): {(cooks > thr).sum()}개 — 이들을 빼면 결론이 바뀌는지 점검하면 좋습니다.")

> **읽는 법(그림):** 임계선 위로 솟은 막대가 회귀선을 크게 흔드는 관측치입니다. 보고서에 "영향점 N개, 제거 시 결론 불변/변동"을 적어두면 신뢰성 평가가 완성됩니다.

## 📋 회귀 분석 보고서 (연관 수준 — 인과 주장 없음)

위 결과를 아래 양식으로 정리하세요.

```markdown
# 회귀 분석 보고서

## 1. 질문과 데이터
- 어떤 지표들이 질병 진행도와 어떻게 연관되는가? (diabetes, n=442)

## 2. 모델과 핵심 결과
- 다중 회귀, R² = __, 조정 R² = __, F-검정 p = __
| 변수 | 계수 | 95% CI | p | 통제 후 연관 |
| --- | --- | --- | --- | --- |
| bmi | __ | __ | __ | 양(+), 유의 |
| bp  | __ | __ | __ | ... |

## 3. 신뢰성 점검
- 다중공선성(VIF): 최대 __ (문제/양호)
- 잔차 진단: 패턴/정규성 (양호/주의)
- 영향점: Cook's distance 초과 __개

## 4. 결론과 한계  ← 인과 금지
- (연관) "다른 지표를 통제했을 때 bmi가 높을수록 진행도가 높은 *경향*"
- (한계) 관찰 데이터 → **인과 아님**. 빠진 교란 가능. 인과는 실험 설계 필요.
```

> 💡 빈칸(`__`)은 위 출력으로 채우세요. 특히 **4번에서 "일으킨다"가 아니라 "연관된다"로 쓰는 것** — 오늘 보고서의 가장 중요한 약속입니다.

## 통제의 위력을 한 그림으로 — 심슨의 역설

회귀의 '통제'가 왜 그토록 중요한지 보여주는 유명한 사례가 **심슨의 역설(Simpson's paradox)**입니다. *전체*로 보면 양의 관계인데, *집단을 나눠* 보면 모든 집단에서 음의 관계로 **방향이 뒤집히는** 현상입니다. 숨은 집단(교란)을 무시하면 정반대 결론에 도달할 수 있다는 강력한 경고입니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/47/Simpson%27s_paradox_continuous.svg" width="560" alt="Simpson's paradox continuous.svg">

심슨의 역설. 데이터 전체에 직선을 맞추면 **우상향**(점선)이지만, 색으로 구분된 집단 안에서는 모두 **우하향**입니다. 어느 쪽이 맞을까요 — 질문에 따라 다르지만, 적어도 집단을 무시한 결론은 위험합니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Simpson's paradox continuous.svg` (Public domain) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Simpson%27s_paradox_continuous.svg

> 💡 **CASchools 사례와 같은 구조입니다.** 거기서는 "학급이 큰 학군일수록 저소득"이라는 숨은 구조가 계수를 4배로 부풀렸습니다. 심슨의 역설은 그 극단적인 형태 — 부풀리기를 넘어 *부호까지 뒤집는* 경우입니다.

In [ ]:
# [C28] 심슨의 역설 — 전체는 양(+), 집단 안에서는 음(-)
np.random.seed(1)                                                 # 재현성: 난수를 항상 같게 고정
xs, ys, gs = [], [], []
for k, (cx, cy) in enumerate([(1, 2), (3, 4), (5, 6), (7, 8)]):   # 집단 중심은 우상향(전체 +)
    gx = np.random.normal(cx, 0.5, 40)
    gy = cy - 0.8 * (gx - cx) + np.random.normal(0, 0.25, 40)      # 집단 안은 음의 기울기
    xs += gx.tolist(); ys += gy.tolist(); gs += [k] * 40
xs, ys, gs = np.array(xs), np.array(ys), np.array(gs)

r_all = stats.pearsonr(xs, ys)[0]                                 # Pearson 상관 — '직선' 관계의 세기 (-1~1)
fig, ax = plt.subplots(figsize=(7.5, 5))                          # 그래프 하나를 그릴 도화지 준비
ax.scatter(xs, ys, c=gs, cmap="viridis", alpha=0.6, s=22)
# 전체 회귀선 (양)
b1, b0 = np.polyfit(xs, ys, 1)
gxr = np.array([xs.min(), xs.max()]); ax.plot(gxr, b0 + b1*gxr, "r--", lw=2, label=f"overall (slope {b1:+.2f})")
# 집단별 회귀선 (음)
for k in range(4):
    mask = gs == k
    bb1, bb0 = np.polyfit(xs[mask], ys[mask], 1)
    gg = np.array([xs[mask].min(), xs[mask].max()]); ax.plot(gg, bb0 + bb1*gg, color="gray", lw=1.5)
ax.set_title(f"Simpson's paradox: overall r = {r_all:+.2f}, but each group is negative")
ax.legend(); plt.tight_layout(); plt.show()                       # 그래프 간격 자동 정리 후 출력
print(f"전체 상관 = {r_all:+.2f} (양)  /  각 집단 안에서는 모두 음(-)의 관계")

> **읽는 법(그림):** 빨간 점선(전체)은 우상향(양의 관계)인데, 회색 선들(집단별)은 모두 우하향(음의 관계)입니다. 집단(색)이라는 숨은 변수를 무시하면 "양의 관계"라 결론짓지만, 집단을 *통제*하면 진실은 정반대입니다. 다중 회귀에 그 집단 변수를 넣으면 바로 이 '통제'를 해냅니다 — Part 6의 통제가 왜 본질적인지 보여주는 그림입니다.

## ⌨️ 백문이 불여일타! (5) — 변수 조합별 조정 R² 실험

```
[문제]
1) 위 종합 회귀에서 설명변수를 자유롭게 바꿔(추가·제거) 조정 R²가 어떻게 변하는지 실험해보세요.
2) 변수를 무작정 늘리는 게 항상 좋은지 결과로 확인해보세요.
```

In [ ]:
# [C29] ⌨️ 백문이 불여일타! (5) — 변수 조합별 조정 R² 실험

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C24]의 `feat` 리스트를 여러 조합으로 바꿔가며 다시 적합해보세요.
- 비교 기준은 `rsquared`가 아니라 `rsquared_adj`입니다 — 둘을 나란히 출력하면 차이가 보입니다.
- 변수를 늘렸는데 조정 R²가 오히려 떨어지는 조합이 있는지 찾아보세요.

</details>

## 결과를 제대로 보고하는 법 — 표준 표기와 흔한 실수

회귀 결과도 **표준 형식**으로 보고합니다(학계·실무 공통).

| 항목 | 표준 보고 형식 (예) |
|------|---------------------|
| 단순/다중 회귀 계수 | "BMI는 진행도와 양의 연관, `β = 537.6, 95% CI [408.8, 666.3], p < .001`" |
| 모델 적합 | "다중 회귀가 변동의 51%를 설명, `R² = .51, 조정 R² = .50, F(7,434) = 65.4, p < .001`" |
| 진단 | "VIF 최대 1.5(양호), 잔차 정규성·등분산 큰 위반 없음" |

> 📌 **흔한 실수 5가지 (제출 전 체크):**
> 1. 관찰 데이터 회귀를 **인과로 서술** → "일으킨다"가 아니라 "연관된다"로 (오늘의 1번 약속)
> 2. **$R^2$만 보고 모델을 평가** → 계수의 유의성·진단·맥락을 함께 봐야 합니다
> 3. **다중공선성(VIF)을 점검 안 함** → 얽힌 변수의 계수는 부호조차 못 믿습니다
> 4. **잔차 진단 생략** → 가정이 깨지면 p값·신뢰구간이 틀어집니다
> 5. **계수를 "통제" 없이 해석** → 다중 회귀 계수는 *다른 변수가 같을 때*의 효과입니다

> 💡 **한 문장 원칙:** *상관은 산점도와 함께, 회귀계수는 신뢰구간·진단과 함께, 그리고 결론은 늘 "연관"의 언어로.*

# ❓ 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다.

## 개념 퀴즈

1. 곡선(비선형) 관계인데 Pearson r이 0에 가깝게 나왔습니다. 무엇을 함께 봐야 했을까요?
2. "아이스크림 판매와 익사가 강한 양의 상관" — 여기서 교란변수는 무엇일까요?
3. 다중 회귀에서 어떤 변수의 계수는 "______를 통제했을 때"의 효과입니다. 빈칸을 채우세요.
4. VIF가 매우 큰(예: 100) 변수가 있으면 어떤 문제를 의심하나요?

## 코드 퀴즈

diabetes에서 **bmi, bp 두 변수로** 다중 회귀를 적합하고, $R^2$와 두 계수의 p값을 출력하세요.

In [ ]:
# [C30] 코드 퀴즈 — bmi·bp 다중 회귀 적합과 R²·p값 출력

# 여기에 코드를 작성하세요

# 🎓 오늘의 정리

## 오늘 배운 것의 연결

```text
[Part 3] 상관 — 함께 움직이는가 (한 숫자)
   ↓ 그런데 상관은
[Part 4] 상관 ≠ 인과 (교란변수의 함정)
   ↓ 관계를 '식'으로
[Part 5] 단순 회귀 (한 변수, OLS, R²)
   ↓ 여러 변수로, '통제'
[Part 6] 다중 회귀 (통제된 효과, F, 조정 R²)
   ↓ 믿어도 되나?
[Part 7] 진단 (VIF·잔차·이상치)
   ↓ 그래서 회귀란
[Part 8] 세 얼굴 (예측·검정·인과 도구) → 다음 시간으로
```

## 한 장 정리

| 개념 | 한 줄 요약 |
|------|-----------|
| 상관계수 | 두 변수가 함께 움직이는 방향·세기 (−1~1) |
| 상관 ≠ 인과 | 교란변수가 만든 허위상관을 조심 |
| 단순 회귀 | $\hat{y}=\beta_0+\beta_1x$, 기울기 = x 1당 y 변화 |
| 다중 회귀 | 계수 = *다른 변수 통제 후* 평균 효과 |
| $R^2$ / 조정 $R^2$ | 설명된 변동 비율 / 변수 수 보정판 |
| VIF | 다중공선성 지표 (5~10↑ 의심) |
| 잔차 진단 | 무패턴·정규·등분산이면 건강 |

## 진단 → 다음 분석

| 상황 | 다음 단계 |
|------|-----------|
| "이 연관이 진짜 *원인*인가" 알고 싶다 | A/B 실험·인과추론 |
| 실험을 *할 수 없는* 관찰 데이터다 | 준실험 설계 — DiD·PSM |
| 예측 정확도를 더 높이고 싶다 | 정규화·트리·교차검증 (이후 ML) |

> 💡 **다음 시간 예고:** 오늘 우리는 회귀로 교란을 *조금* 통제했지만, "측정 못 한 교란"은 여전히 남았습니다. 그래서 다음 시간엔 **무작위 실험(A/B 테스트)과 인과추론**으로 — "우리가 만든 변화가 *진짜* 원인인가"를 정면으로 묻습니다. 오늘의 "상관 ≠ 인과"가 그 출발점입니다.

# 📝 오늘의 과제

> 짧은 점검은 노트북에서 끝냈습니다. 이제 **본인이 고른 데이터에 본격 적용**하는 과제입니다.

## 과제 — 다중 회귀 분석 보고서

1. **데이터:** 아래처럼 **라이브러리가 제공하는 데이터셋** 중, *연속형 종속변수*가 있는 것을 하나 고르세요.

   ```python
   import seaborn as sns
   import statsmodels.api as sm

   # (a) seaborn
   sns.load_dataset("mpg")          # mpg(연비) ~ horsepower·weight·model_year·origin
   sns.load_dataset("diamonds")     # price ~ carat·cut·color·clarity  (표본 큼, 로그 변환 연습에 좋음)
   sns.load_dataset("penguins")     # body_mass_g ~ flipper_length_mm·bill_length_mm·species·sex

   # (b) statsmodels 내장
   sm.datasets.engel.load_pandas().data          # foodexp ~ income
   sm.datasets.statecrime.load_pandas().data     # violent ~ poverty·hs_grad·single·urban
   sm.datasets.macrodata.load_pandas().data      # realgdp 등 거시 지표

   # (c) Rdatasets
   sm.datasets.get_rdataset("CASchools", "AER").data      # 시험 점수 ~ 학생-교사비·배경
   sm.datasets.get_rdataset("HousePrices", "AER").data    # price ~ lotsize·bedrooms·bathrooms…
   sm.datasets.get_rdataset("Wage", "ISLR").data          # wage ~ age·education·year
   ```

   > 💡 **범주형 변수가 있는 데이터를 고르면 더 좋습니다** — 더미 변수(예: `origin`, `species`, `cut`)를 넣어 해석해보는 연습까지 할 수 있습니다. `smf.ols("y ~ x + C(범주형)", data=...)`처럼 `C()`로 감싸면 더미가 자동 생성됩니다.

2. **분석:**
   - 상관 행렬·**산점도**로 탐색한 뒤(앤스컴의 교훈!), **다중 회귀**를 적합하세요.
   - 계수·신뢰구간·p값·$R^2$·F-검정을 해석하세요. **계수 해석에는 단위를 반드시 붙이세요**("면적이 1㎡ 늘면 가격이 평균 __만 원 높다").
   - 관심 변수 하나를 골라 **단순 회귀 vs 다중 회귀 계수를 비교**하고, 차이를 '통제' 개념으로 설명하세요(본문 `CASchools` 사례처럼).
   - **VIF·잔차 진단·이상치(Cook's distance)·Breusch-Pagan**으로 모델 신뢰성을 점검하고, 이분산이면 **robust SE(HC3)** 로 다시 보고하세요.
3. **보고서:** 위 종합 실습 양식으로 정리하되, **결론은 "연관" 수준으로만** 적습니다(인과 주장 금지). 빠졌을 법한 교란변수도 한 줄 적어보세요.

### 심화 (선택)

- [ ] 종속변수(또는 설명변수)에 **로그 변환**을 적용해 계수 해석이 어떻게 바뀌는지(탄력성) 비교했다.
- [ ] 상호작용항(`x1:x2`)을 넣어 "효과가 집단마다 다른가"를 확인했다.
- [ ] 조정 $R^2$로 변수 조합을 비교하고, 변수를 늘리는 것이 늘 좋지 않음을 보였다.

---

오늘도 한 걸음, 수고하셨습니다! 🎉
어제의 나보다 데이터를 다루는 손이 한 뼘 더 능숙해졌습니다. 다음 시간에 또 만나요.

오늘 여러분은 데이터 분석의 주력 도구 — **회귀**를 손에 넣었습니다. 상관에서 출발해, 한 변수·여러 변수로 결과를 설명하고, 그 모델을 진단하고, 무엇보다 **"연관과 인과를 구분하는 신중함"**을 배웠습니다. 다음 시간엔 그 인과를 정면으로 다루는 **실험과 인과추론**으로 넘어갑니다. 천천히, 그러나 멈추지 않고 가봅시다. 🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>